# Ampliación CUDA: FC-A y FC-B

Ejecución nueva y nativa, exclusiva de estas dos redes. Cinco corridas por arquitectura. Las campañas CPU y CNN-B anteriores **no se ejecutan**. Preparación y validación en CPU; entrenamiento matemático en CUDA Python/cuBLAS. Datos e inicialización históricos verificados. Los contadores pertenecen a esta sesión nueva, distinta del notebook CNN-B previo.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, time, csv, gc, subprocess
ROOT=Path.cwd()
while not (ROOT/'docs/Proyecto 3 CUDA.pdf').is_file():
    if ROOT==ROOT.parent: raise FileNotFoundError('Raíz del proyecto')
    ROOT=ROOT.parent
sys.path[:0]=[str(ROOT/'src'),str(ROOT/'scripts')]
for key,value in {'OMP_NUM_THREADS':'6','MKL_NUM_THREADS':'6','OPENBLAS_NUM_THREADS':'6',
    'TF_NUM_INTRAOP_THREADS':'6','TF_NUM_INTEROP_THREADS':'1','TF_DETERMINISTIC_OPS':'1',
    'TF_ENABLE_ONEDNN_OPTS':'1','NUMBA_CUDA_LOW_OCCUPANCY_WARNINGS':'0'}.items():os.environ[key]=value
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,Image
from threadpoolctl import threadpool_limits
from benchmark_cpu import sha,write_json,write_csv,utc
from benchmark_cuda import load_manifest,load_images
from registro_notebook import iniciar_monitor,mostrar_figura
CONFIG=ROOT/'configuraciones/EXP-012.json'
config=json.loads(CONFIG.read_text(encoding='utf8'))
for relative,digest in config['codigo']['fuentes'].items():
    assert sha(ROOT/relative)==digest,relative
assert sha(ROOT/config['validacion']['ruta'])==config['validacion']['sha256']
assert json.loads((ROOT/config['validacion']['ruta']).read_text(encoding='utf8'))['estado']=='aprobado'
manifest=load_manifest(config)
data_root=Path(config['datos']['ruta'])
for split in manifest.values():
    for row in split:
        assert sha(data_root/row['ruta_relativa'])==row['sha256'],row['ruta_relativa']
limits=threadpool_limits(limits=6);limits.__enter__()
print('EXP-012 | SOLO FC-A / FC-B CUDA | CPU y CNN-B anteriores NO ejecutados')
print('Configuración SHA-256:',sha(CONFIG),'Python:',sys.executable)
print(json.dumps(config,indent=2,ensure_ascii=False))


## CUDA Python: código efectivamente definido en esta sesión

Se muestran las dependencias reutilizadas, sin instanciar ni entrenar CNN-B. El último módulo contiene la extensión FC y la corrección de momentum BN.

In [ ]:
from IPython.core.magic import register_cell_magic
import types
@register_cell_magic
def modulo_fc(line,cell):
    name=line.strip();path=ROOT/'src'/Path(*name.split('.')).with_suffix('.py')
    module=types.ModuleType(name);module.__file__=str(path);module.__package__=name.rpartition('.')[0]
    sys.modules[name]=module
    exec(compile(cell,str(path),'exec'),module.__dict__)
    parent,_,attr=name.rpartition('.')
    if parent in sys.modules:setattr(sys.modules[parent],attr,module)
    print('Módulo CUDA Python definido:',name,'SHA-256',sha(path))
import asl_cuda


In [ ]:
%%modulo_fc asl_cuda.runtime
"""Local toolkit discovery. Never installs packages or silently falls back to CPU."""
import os
from pathlib import Path

_DLL_HANDLES = []


def configure():
    if os.environ.get("NUMBA_ENABLE_CUDASIM", "0") != "0":
        raise RuntimeError("CUDA simulator is not a GPU benchmark.")
    if os.name == "nt":
        root = Path(os.environ.get("CUDA_HOME", "C:/Program Files/NVIDIA GPU Computing Toolkit/CUDA/v13.1"))
        os.environ.setdefault("CUDA_HOME", str(root))
        for folder in (root / "bin/x64", root / "bin", root / "nvvm/bin/x64", root / "nvvm/bin"):
            if folder.is_dir():
                _DLL_HANDLES.append(os.add_dll_directory(str(folder)))


configure()
# numba-cuda 0.30.4 still registers NumPy's retired row_stack alias at import.
# Preserve the installed NumPy 2.5.5: the alias is exactly vstack, not new math.
# Process-local compatibility only; no site-packages files are changed.
import numpy as np
if not hasattr(np, "row_stack"):
    np.row_stack = np.vstack
from numba import cuda  # noqa: E402


def device_info():
    if not cuda.is_available():
        raise RuntimeError("A real CUDA device is required.")
    d = cuda.get_current_device()
    free, total = cuda.current_context().get_memory_info()
    return dict(name=d.name.decode() if isinstance(d.name, bytes) else d.name,
                compute_capability=list(d.compute_capability), free_bytes=int(free), total_bytes=int(total),
                max_threads_per_block=int(d.MAX_THREADS_PER_BLOCK),
                max_shared_memory_per_block=int(d.MAX_SHARED_MEMORY_PER_BLOCK),
                simulator=False)


In [ ]:
%%modulo_fc asl_cuda.kernels
"""K01-K19, DIS-CUDA-001: FP32 NHWC, gather backward, explicit reductions.

Flat buffers have explicit active dimensions; capacity never defines batch size.
No floating point atomics or cross-block barriers. Partial gradients are unique.
"""
import math
from numba import float32
from .runtime import cuda


@cuda.jit(device=True, inline=True)
def _left(a, mode, m, k, H, W, Ci, Co, I, O):
    if mode == 0:  # virtual im2col
        n = m // (H * W)
        h = (m // W) % H + k // (3 * Ci) - 1
        w = m % W + (k // Ci) % 3 - 1
        if 0 <= h < H and 0 <= w < W:
            return a[((n * H + h) * W + w) * Ci + k % Ci]
        return float32(0)
    if mode == 1:  # gather dX, spatially reversed offsets
        n = m // (H * W)
        h = (m // W) % H - k // (3 * Co) + 1
        w = m % W - (k // Co) % 3 + 1
        if 0 <= h < H and 0 <= w < W:
            return a[((n * H + h) * W + w) * Co + k % Co]
        return float32(0)
    if mode == 2:  # im2col transpose without materializing it
        n = k // (H * W)
        h = (k // W) % H + m // (3 * Ci) - 1
        w = k % W + (m // Ci) % 3 - 1
        if 0 <= h < H and 0 <= w < W:
            return a[((n * H + h) * W + w) * Ci + m % Ci]
        return float32(0)
    if mode == 3:
        return a[m * I + k]
    if mode == 4:
        return a[m * O + k]
    return a[k * I + m]


@cuda.jit(device=True, inline=True)
def _right(v, mode, k, n, Ci, Co, I, O):
    if mode == 0 or mode == 2:
        return v[k * Co + n]
    if mode == 1:
        return v[((k // Co) * Ci + n) * Co + k % Co]
    if mode == 3 or mode == 5:
        return v[k * O + n]
    return v[n * O + k]


def _make_gemm(mode):
    @cuda.jit(cache=True, fastmath=False)
    def kernel(a, v, bias, out, M, N, K, H, W, Ci, Co, I, O, relu, m0):
        sa = cuda.shared.array((16, 16), float32)
        sb = cuda.shared.array((16, 16), float32)
        tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
        m, n = m0 + cuda.blockIdx.y * 16 + ty, cuda.blockIdx.x * 16 + tx
        begin, end = 0, K
        if mode == 2 or mode == 5:
            begin = cuda.blockIdx.z * 2048
            end = min(K, begin + 2048)
        acc = float32(0)
        for k0 in range(begin, end, 16):
            sa[ty, tx] = float32(0)
            sb[ty, tx] = float32(0)
            if m < M and k0 + tx < end:
                sa[ty, tx] = _left(a, mode, m, k0 + tx, H, W, Ci, Co, I, O)
            if n < N and k0 + ty < end:
                sb[ty, tx] = _right(v, mode, k0 + ty, n, Ci, Co, I, O)
            cuda.syncthreads()
            for kk in range(16):
                acc += sa[ty, kk] * sb[kk, tx]
            cuda.syncthreads()
        if m < M and n < N:
            index = m * N + n
            if mode == 2 or mode == 5:
                index += cuda.blockIdx.z * M * N
            if mode == 0 or mode == 3:
                acc += bias[n]
                if relu and acc < 0:
                    acc = float32(0)
            out[index] = acc
    return kernel


GEMM = tuple(_make_gemm(i) for i in range(6))


@cuda.jit(cache=True)
def merge_partials(parts, out, E, P):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        value = float32(0)
        for p in range(P):
            value += parts[p * E + i]
        out[i] = value


@cuda.jit(cache=True)
def copy_flat(x, y, E):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        y[i] = x[i]


@cuda.jit(cache=True)
def relu_backward(grad, activation, E):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        if activation[i] <= 0:
            grad[i] = float32(0)


@cuda.jit(cache=True)
def dropout(x, mask, out, factor, E):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        out[i] = x[i] * float32(mask[i]) * factor


@cuda.jit(cache=True)
def pool_forward(x, y, arg, b, H, W, C):
    E = b * (H // 2) * (W // 2) * C
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        c = i % C
        w = (i // C) % (W // 2)
        h = (i // (C * (W // 2))) % (H // 2)
        n = i // (C * (W // 2) * (H // 2))
        base = ((n * H + 2 * h) * W + 2 * w) * C + c
        value, pos = x[base], 0
        for p in range(1, 4):
            candidate = x[base + (p // 2) * W * C + (p % 2) * C]
            if candidate > value:
                value, pos = candidate, p
        y[i], arg[i] = value, pos


@cuda.jit(cache=True)
def pool_backward(dy, arg, dx, b, H, W, C):
    for i in range(cuda.grid(1), b * H * W * C, cuda.gridsize(1)):
        c, w = i % C, (i // C) % W
        h, n = (i // (C * W)) % H, i // (C * W * H)
        out_i = ((n * (H // 2) + h // 2) * (W // 2) + w // 2) * C + c
        dx[i] = float32(0)
        if arg[out_i] == 2 * (h % 2) + w % 2:
            dx[i] = dy[out_i]


@cuda.jit(cache=True, fastmath=False)
def reduce_channels(x, dy, mean, inv, parts1, parts2, L, C, mode):
    # mode0 sum; mode1 centered variance; mode2 BN backward pair.
    s1 = cuda.shared.array((8, 32), float32)
    s2 = cuda.shared.array((8, 32), float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    c = cuda.blockIdx.y * 32 + tx
    v1, v2 = float32(0), float32(0)
    if c < C:
        for j in range(256):
            l = cuda.blockIdx.x * 2048 + ty + 8 * j
            if l < L:
                value = x[l * C + c]
                if mode == 0:
                    v1 += value
                elif mode == 1:
                    centered = value - mean[c]
                    v1 += centered * centered
                else:
                    g = dy[l * C + c]
                    v1 += g
                    v2 += g * ((value - mean[c]) * inv[c])
    s1[ty, tx], s2[ty, tx] = v1, v2
    cuda.syncthreads()
    stride = 4
    while stride > 0:
        if ty < stride:
            s1[ty, tx] += s1[ty + stride, tx]
            s2[ty, tx] += s2[ty + stride, tx]
        cuda.syncthreads()
        stride //= 2
    if ty == 0 and c < C:
        parts1[cuda.blockIdx.x * C + c] = s1[0, tx]
        parts2[cuda.blockIdx.x * C + c] = s2[0, tx]


@cuda.jit(cache=True, fastmath=False)
def finish_channels(p1, p2, out1, out2, movmean, movvar, mean, inv, P, C, L, mode):
    # mode0 mean; mode1 variance+moving state; mode2 dbeta/dgamma; mode3 bias.
    s1 = cuda.shared.array(256, float32)
    s2 = cuda.shared.array(256, float32)
    t, c = cuda.threadIdx.x, cuda.blockIdx.x
    s1[t], s2[t] = float32(0), float32(0)
    if t < P:
        s1[t], s2[t] = p1[t * C + c], p2[t * C + c]
    cuda.syncthreads()
    stride = 128
    while stride > 0:
        if t < stride:
            s1[t] += s1[t + stride]
            s2[t] += s2[t + stride]
        cuda.syncthreads()
        stride //= 2
    if t == 0:
        if mode == 0:
            out1[c] = s1[0] / float32(L)
        elif mode == 1:
            var = s1[0] / float32(L)
            inv[c] = float32(1) / math.sqrt(var + float32(.001))
            movmean[c] = float32(.9) * movmean[c] + float32(.1) * mean[c]
            movvar[c] = float32(.9) * movvar[c] + float32(.1) * var
        elif mode == 2:
            out1[c], out2[c] = s1[0], s2[0]
        else:
            out1[c] = s1[0]


@cuda.jit(cache=True, fastmath=False)
def bn_forward(x, y, gamma, beta, mean, inv, E, C):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        c = i % C
        y[i] = gamma[c] * ((x[i] - mean[c]) * inv[c]) + beta[c]


@cuda.jit(cache=True, fastmath=False)
def bn_backward(x, dy, dx, gamma, mean, inv, dbeta, dgamma, L, C):
    for i in range(cuda.grid(1), L * C, cuda.gridsize(1)):
        c = i % C
        xhat = (x[i] - mean[c]) * inv[c]
        dx[i] = (gamma[c] * inv[c] / float32(L)) * (
            float32(L) * dy[i] - dbeta[c] - xhat * dgamma[c])


@cuda.jit(cache=True, fastmath=False)
def softmax_loss(logits, labels, probs, dlogits, loss, correct, b):
    s = cuda.shared.array(32, float32)
    t, n = cuda.threadIdx.x, cuda.blockIdx.x
    z = float32(-math.inf)
    if t < 29:
        z = logits[n * 29 + t]
    s[t] = z
    cuda.syncthreads()
    stride = 16
    while stride > 0:
        if t < stride:
            s[t] = max(s[t], s[t + stride])
        cuda.syncthreads()
        stride //= 2
    maximum = s[0]
    cuda.syncthreads()
    value = float32(0)
    if t < 29:
        value = math.exp(z - maximum)
    s[t] = value
    cuda.syncthreads()
    stride = 16
    while stride > 0:
        if t < stride:
            s[t] += s[t + stride]
        cuda.syncthreads()
        stride //= 2
    denominator = s[0]
    label = labels[n]
    if t < 29:
        probability = value / denominator
        probs[n * 29 + t] = probability
        dlogits[n * 29 + t] = (probability - float32(t == label)) / float32(b)
    if t == 0:
        loss[n] = (maximum - logits[n * 29 + label]) + math.log(denominator)
        winner = 0
        for c in range(1, 29):
            if logits[n * 29 + c] > logits[n * 29 + winner]:
                winner = c
        correct[n] = winner == label


@cuda.jit(cache=True, fastmath=False)
def adam(theta, grad, moment, velocity, alpha, E, error):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        g = grad[i]
        if not math.isfinite(g):
            cuda.atomic.max(error, 0, 1)
        if error[0] == 0:
            m = moment[i] + (g - moment[i]) * float32(.1)
            v = velocity[i] + (g * g - velocity[i]) * float32(.001)
            value = theta[i] - (m * alpha) / (math.sqrt(v) + float32(1e-7))
            if not math.isfinite(value):
                cuda.atomic.max(error, 0, 2)
            theta[i], moment[i], velocity[i] = value, m, v


@cuda.jit(cache=True)
def finite_check(x, E, error):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        if not math.isfinite(x[i]):
            cuda.atomic.max(error, 0, 3)


@cuda.jit(cache=True)
def metrics(loss, correct, b, total):
    s = cuda.shared.array(256, float32)
    q = cuda.shared.array(256, float32)
    t = cuda.threadIdx.x
    s[t], q[t] = float32(0), float32(0)
    if t < b:
        s[t], q[t] = loss[t], float32(correct[t])
    cuda.syncthreads()
    stride = 128
    while stride > 0:
        if t < stride:
            s[t] += s[t + stride]
            q[t] += q[t + stride]
        cuda.syncthreads()
        stride //= 2
    if t == 0:
        total[0] += s[0]
        total[1] += q[0]
        total[2] += float32(b)


@cuda.jit(cache=True)
def advance_step(counter):
    if cuda.grid(1) == 0:
        counter[0] += 1


@cuda.jit(cache=True)
def im2col(x, cols, R, H, W, Ci):
    K = 9 * Ci
    for i in range(cuda.grid(1), R * K, cuda.gridsize(1)):
        r, kk = i // K, i % K
        n = r // (H * W)
        h = (r // W) % H + kk // (3 * Ci) - 1
        w = r % W + (kk // Ci) % 3 - 1
        value = float32(0)
        if 0 <= h < H and 0 <= w < W:
            value = x[((n * H + h) * W + w) * Ci + kk % Ci]
        cols[i] = value


@cuda.jit(cache=True)
def col2im_gather(cols, dx, R, H, W, Ci):
    for i in range(cuda.grid(1), R * Ci, cuda.gridsize(1)):
        c = i % Ci
        w, h = (i // Ci) % W, (i // (Ci * W)) % H
        n = i // (Ci * W * H)
        value = float32(0)
        for kh in range(3):
            for kw in range(3):
                ho, wo = h - kh + 1, w - kw + 1
                if 0 <= ho < H and 0 <= wo < W:
                    row = (n * H + ho) * W + wo
                    value += cols[row * 9 * Ci + (kh * 3 + kw) * Ci + c]
        dx[i] = value


@cuda.jit(cache=True)
def bias_activation(y, bias, E, C, relu):
    for i in range(cuda.grid(1), E, cuda.gridsize(1)):
        value = y[i] + bias[i % C]
        if relu and value < 0:
            value = float32(0)
        y[i] = value


In [ ]:
%%modulo_fc asl_cuda.blas
"""FP32 cuBLAS SGEMM on Numba-owned device memory and the SAME CUDA stream.

The installed NVIDIA CUDA toolkit supplies the DLL. No tensor cores, no TF32,
no NumPy/CPU multiplication. Geometry inside cuBLAS belongs to NVIDIA's backend.
"""
import ctypes as ct
import os
from pathlib import Path


class Blas:
    def __init__(self, stream):
        root = Path(os.environ["CUDA_HOME"])
        self.path = root / "bin/x64/cublas64_13.dll"
        self.dll = ct.WinDLL(str(self.path))
        self.handle = ct.c_void_p()
        self._signature("cublasCreate_v2", [ct.POINTER(ct.c_void_p)])
        self._signature("cublasSetStream_v2", [ct.c_void_p, ct.c_void_p])
        self._signature("cublasSetMathMode", [ct.c_void_p, ct.c_int])
        self._signature("cublasSetAtomicsMode", [ct.c_void_p, ct.c_int])
        self._signature("cublasGetVersion_v2", [ct.c_void_p, ct.POINTER(ct.c_int)])
        self._signature("cublasDestroy_v2", [ct.c_void_p])
        self._signature("cublasSgemm_v2", [ct.c_void_p, ct.c_int, ct.c_int,
            ct.c_int, ct.c_int, ct.c_int, ct.POINTER(ct.c_float), ct.c_void_p, ct.c_int,
            ct.c_void_p, ct.c_int, ct.POINTER(ct.c_float), ct.c_void_p, ct.c_int])
        self._call("cublasCreate_v2", ct.byref(self.handle))
        raw = stream.handle
        self._call("cublasSetStream_v2", self.handle, ct.c_void_p(raw.value if hasattr(raw, "value") else int(raw)))
        self._call("cublasSetMathMode", self.handle, 2)  # CUBLAS_PEDANTIC_MATH
        self._call("cublasSetAtomicsMode", self.handle, 0)  # CUBLAS_ATOMICS_NOT_ALLOWED
        version = ct.c_int()
        self._call("cublasGetVersion_v2", self.handle, ct.byref(version))
        self.version = version.value

    def _signature(self, name, args):
        fn = getattr(self.dll, name)
        fn.argtypes, fn.restype = args, ct.c_int

    def _call(self, name, *args):
        status = getattr(self.dll, name)(*args)
        if status:
            raise RuntimeError(f"{name}: cuBLAS status {status}")

    @staticmethod
    def ptr(array):
        return ct.c_void_p(array.device_ctypes_pointer.value)

    def gemm(self, a, b, out, M, N, K, transpose_a=False, transpose_b=False):
        """C_row=op(A)_row @ op(B)_row, implemented as C_col=B_col @ A_col."""
        alpha, beta = ct.c_float(1.), ct.c_float(0.)
        self._call("cublasSgemm_v2", self.handle, int(transpose_b), int(transpose_a),
                   N, M, K, ct.byref(alpha), self.ptr(b), K if transpose_b else N,
                   self.ptr(a), M if transpose_a else K, ct.byref(beta), self.ptr(out), N)

    def close(self):
        if self.handle:
            self._call("cublasDestroy_v2", self.handle)
            self.handle = ct.c_void_p()


In [ ]:
%%modulo_fc asl_cuda.model
"""Complete CNN-B training on explicitly launched CUDA Python kernels.

TensorFlow is used only as a CPU oracle and for CPU validation/checkpoint I/O.
The original network and its initial state are imported without editing them.
"""
from contextlib import contextmanager
from pathlib import Path
import hashlib
import json
import time
import numpy as np
from .runtime import cuda, device_info
from . import kernels as k

ROOT = Path(__file__).resolve().parents[2]
INITIAL = ROOT / "data/EXP-002/RUN-20261117T171942Z-cpu-01/CNN-B.estado-inicial.npz"
INITIAL_SHA = "10558d723213752b759626c3b196665f2547a0b65cbce296e47b3a4f992305fd"


def cpu_reference():
    """Build original definitions, confirm CPU placement and restore ALL 80 arrays."""
    import os
    for name, value in {"TF_NUM_INTRAOP_THREADS":"6", "TF_NUM_INTEROP_THREADS":"1",
                        "OMP_NUM_THREADS":"6", "OPENBLAS_NUM_THREADS":"6",
                        "MKL_NUM_THREADS":"6", "TF_DETERMINISTIC_OPS":"1",
                        "TF_ENABLE_ONEDNN_OPTS":"1"}.items():
        os.environ[name] = value
    import tensorflow as tf
    tf.config.set_visible_devices([], "GPU")
    tf.config.threading.set_intra_op_parallelism_threads(6)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.keras.utils.set_random_seed(43)
    tf.keras.mixed_precision.set_global_policy("float32")
    import sys
    sys.path.insert(0, str(ROOT / "scripts"))
    from benchmark_cpu import original_namespace
    ns = original_namespace()
    ns.update(keras=tf.keras, layers=tf.keras.layers, NUM_CLASSES=29)
    model = ns["build_cnn_b"]()
    model.jit_compile = False
    model.optimizer.build(model.trainable_variables)
    if hashlib.sha256(INITIAL.read_bytes()).hexdigest() != INITIAL_SHA:
        raise ValueError("Initial checkpoint identity changed")
    with np.load(INITIAL, allow_pickle=False) as archive:
        variables = list(model.variables) + list(model.optimizer.variables)
        if len(variables) != len(archive.files):
            raise ValueError("Initial state variable count differs")
        for index, variable in enumerate(variables):
            value = archive[f"arr_{index}"]
            if tuple(variable.shape) != value.shape or str(variable.dtype) != str(value.dtype):
                raise ValueError(f"Incompatible state variable {index}")
            variable.assign(value)
    if int(model.optimizer.iterations.numpy()) != 0:
        raise ValueError("Expected optimizer iteration zero")
    if any(np.any(v.numpy()) for v in [*model.optimizer._momentums, *model.optimizer._velocities]):
        raise ValueError("Nonzero initial Adam requires explicit state mapping")
    return tf, model


def grid(E):
    return max(1, min(4096, (int(E) + 255) // 256))


def compiled_resources():
    """Numba-CUDA 0.30 API expects signature objects, not .signatures tuples."""
    result={}
    for name,kern in vars(k).items():
        if hasattr(kern,"get_regs_per_thread") and kern.signatures:
            regs=kern.get_regs_per_thread()
            shared=kern.get_shared_mem_per_block()
            local=kern.get_local_mem_per_thread()
            result[name]=[dict(firma=str(sig),registros=value,shared_bytes=shared[sig],local_bytes=local[sig])
                          for sig,value in regs.items()]
    return result


class Network:
    def __init__(self, reference, backend="explicit"):
        self.info = device_info()
        if self.info["free_bytes"] < 1536 * 2**20:
            raise MemoryError("Insufficient free VRAM for the frozen profile")
        self.stream = cuda.stream()
        if backend not in ("explicit", "cublas"):
            raise ValueError("Unknown backend")
        self.backend = backend
        self.allocations, self.params, self.ops, self.transfers = [], [], [], []
        self.reference = reference
        self.input = self.alloc(128 * 64 * 64 * 3)
        self.host_x = cuda.pinned_array(128 * 64 * 64 * 3, dtype=np.float32)
        self.labels = self.alloc(128, np.int64)
        self.host_y = cuda.pinned_array(128, dtype=np.int64)
        shape, previous = (64, 64, 3), self.input
        for layer in reference.layers:
            name = type(layer).__name__
            item = dict(input=previous, shape=shape, layer=layer)
            if name == "Conv2D":
                H, W, Ci = shape
                Co = layer.filters
                item.update(kind="conv", H=H, W=W, Ci=Ci, Co=Co,
                            weights=self.param(layer.kernel), bias=self.param(layer.bias), relu=True)
                shape = (H, W, Co)
            elif name == "Dense":
                I, O = int(layer.kernel.shape[0]), int(layer.kernel.shape[1])
                item.update(kind="dense", I=I, O=O, relu=O != 29,
                            weights=self.param(layer.kernel), bias=self.param(layer.bias))
                shape = (O,)
            elif name == "BatchNormalization":
                C = shape[-1]
                item.update(kind="bn", C=C, gamma=self.param(layer.gamma), beta=self.param(layer.beta),
                            moving_mean=self.alloc(C), moving_var=self.alloc(C),
                            initial_mean=layer.moving_mean.numpy().copy(), initial_var=layer.moving_variance.numpy().copy(),
                            mean=self.alloc(C), inv=self.alloc(C))
            elif name == "MaxPooling2D":
                H, W, C = shape
                item.update(kind="pool", H=H, W=W, C=C)
                shape = (H // 2, W // 2, C)
                item["arg"] = self.alloc(128 * int(np.prod(shape)), np.uint8)
            elif name == "Dropout":
                E = 128 * int(np.prod(shape))
                item.update(kind="drop", p=layer.rate, mask=self.alloc(E, np.uint8),
                            host_mask=cuda.pinned_array(E, dtype=np.uint8))
            elif name == "Flatten":
                shape = (int(np.prod(shape)),)
                continue
            elif name == "InputLayer":
                continue
            else:
                raise TypeError(f"Unsupported original layer {name}")
            item["sample_elements"] = int(np.prod(shape))
            item["output"] = self.alloc(128 * item["sample_elements"])
            self.ops.append(item)
            previous = item["output"]
        self.probs = self.alloc(128 * 29)
        self.loss, self.correct = self.alloc(128), self.alloc(128, np.uint8)
        self.g1, self.g2 = self.alloc(128 * 64 * 64 * 32), self.alloc(128 * 64 * 64 * 32)
        self.parts = self.alloc(16 * 2**20 // 4)
        self.p1, self.p2 = self.alloc(65536), self.alloc(65536)
        self.dummy = self.alloc(256)
        self.total = self.alloc(3)
        self.error = self.alloc(1, np.int32)
        self.counter = self.alloc(1, np.int64)
        self.host_error = cuda.pinned_array(1, dtype=np.int32)
        self.host_total = cuda.pinned_array(3, dtype=np.float32)
        if backend == "cublas":
            from .blas import Blas
            self.blas = Blas(self.stream)
            self.columns = self.alloc(128 * 64 * 64 * (9 * 32))
        self.bytes_allocated = sum(a.nbytes for a in self.allocations)
        if self.bytes_allocated > (1408 if backend == "cublas" else 768) * 2**20:
            raise MemoryError(f"Arena exceeds design: {self.bytes_allocated}")
        if sum(p["value"].size for p in self.params) != 2245245:
            raise ValueError("Architecture parameter count changed")
        self.events = []
        self.restore()

    def alloc(self, count, dtype=np.float32):
        array = cuda.device_array(int(count), dtype=dtype, stream=self.stream)
        self.allocations.append(array)
        return array

    def param(self, variable):
        values = variable.numpy().copy()
        p = dict(initial=values, value=self.alloc(values.size), grad=self.alloc(values.size),
                 m=self.alloc(values.size), v=self.alloc(values.size), variable=variable)
        self.params.append(p)
        return p

    def restore(self):
        for p in self.params:
            p["value"].copy_to_device(p["initial"].ravel(), stream=self.stream)
            for state in ("m", "v"):
                p[state].copy_to_device(np.zeros(p["initial"].size, np.float32), stream=self.stream)
        for op in self.ops:
            if op["kind"] == "bn":
                op["moving_mean"].copy_to_device(op["initial_mean"], stream=self.stream)
                op["moving_var"].copy_to_device(op["initial_var"], stream=self.stream)
        for a in (self.error, self.counter, self.total):
            a.copy_to_device(np.zeros(a.size, dtype=a.dtype), stream=self.stream)
        self.stream.synchronize()
        self.steps = 0
        self.events = []

    @contextmanager
    def timed(self, phase):
        start, end = cuda.event(timing=True), cuda.event(timing=True)
        start.record(self.stream)
        yield
        end.record(self.stream)
        self.events.append((phase, start, end))

    def drain_events(self):
        self.stream.synchronize()
        result = {}
        for phase, start, end in self.events:
            result[phase] = result.get(phase, 0.) + cuda.event_elapsed_time(start, end) / 1000.
        self.events.clear()
        return result

    def gemm(self, mode, a, v, bias, out, M, N, K, H=1, W=1, Ci=1, Co=1, I=1, O=1, relu=False):
        if self.backend == "cublas":
            if mode in (0, 2):
                R = M if mode == 0 else K
                width = 9 * Ci
                k.im2col[grid(R * width), 256, self.stream](a, self.columns, R, H, W, Ci)
                self.blas.gemm(self.columns, v, out, M, N, K, transpose_a=mode == 2)
            elif mode == 1:
                self.blas.gemm(a, v, self.columns, M, 9 * Ci, Co, transpose_b=True)
                k.col2im_gather[grid(M * Ci), 256, self.stream](self.columns, out, M, H, W, Ci)
            else:
                self.blas.gemm(a, v, out, M, N, K, transpose_a=mode == 5, transpose_b=mode == 4)
            if mode in (0, 3):
                k.bias_activation[grid(M * N), 256, self.stream](out, bias, M * N, N, relu)
            return
        P = (K + 2047) // 2048 if mode in (2, 5) else 1
        if mode in (2, 5) and P * M * N > self.parts.size:
            raise MemoryError("Partial gradient workspace exhausted")
        for m0 in range(0, M, 16384):
            geom = ((N + 15) // 16, (min(16384, M - m0) + 15) // 16, P)
            k.GEMM[mode][geom, (16, 16), self.stream](
                a, v, bias, out, M, N, K, H, W, Ci, Co, I, O, relu, m0)

    def reduce(self, x, dy, op, L, C, mode, out1, out2):
        P = (L + 2047) // 2048
        if P > 256 or P * C > self.p1.size:
            raise ValueError("Reduction shape exceeds frozen profile")
        mean, inv = op.get("mean", self.dummy), op.get("inv", self.dummy)
        k.reduce_channels[(P, (C + 31) // 32), (32, 8), self.stream](
            x, dy, mean, inv, self.p1, self.p2, L, C, 2 if mode == 2 else (1 if mode == 1 else 0))
        k.finish_channels[C, 256, self.stream](self.p1, self.p2, out1, out2,
            op.get("moving_mean", self.dummy), op.get("moving_var", self.dummy),
            mean, inv, P, C, L, mode)

    def upload(self, x, labels, epoch, batch_index):
        b = len(x)
        np.copyto(self.host_x[:x.size], x.ravel())
        np.copyto(self.host_y[:b], labels)
        drops = [op for op in self.ops if op["kind"] == "drop"]
        for d, op in enumerate(drops):
            E = b * op["sample_elements"]
            rng = np.random.Generator(np.random.PCG64(np.random.SeedSequence([43, epoch, batch_index, d + 1])))
            op["host_mask"][:E] = rng.random(E, dtype=np.float32) >= np.float32(op["p"])
        with self.timed("h2d_lotes"):
            self.input[:x.size].copy_to_device(self.host_x[:x.size], stream=self.stream)
            self.labels[:b].copy_to_device(self.host_y[:b], stream=self.stream)
            for op in drops:
                E = b * op["sample_elements"]
                op["mask"][:E].copy_to_device(op["host_mask"][:E], stream=self.stream)
        self.transfers.append(dict(batch=b, imagenes_bytes=x.nbytes, etiquetas_bytes=labels.nbytes,
                                   mascaras_bytes=sum(b * op["sample_elements"] for op in drops)))

    def forward(self, b):
        for op in self.ops:
            kind, x, y, E = op["kind"], op["input"], op["output"], b * op["sample_elements"]
            if kind == "conv":
                H, W, Ci, Co = (op[z] for z in ("H", "W", "Ci", "Co"))
                self.gemm(0, x, op["weights"]["value"], op["bias"]["value"], y,
                          b * H * W, Co, 9 * Ci, H, W, Ci, Co, relu=True)
            elif kind == "dense":
                self.gemm(3, x, op["weights"]["value"], op["bias"]["value"], y,
                          b, op["O"], op["I"], I=op["I"], O=op["O"], relu=op["relu"])
            elif kind == "bn":
                C, L = op["C"], E // op["C"]
                self.reduce(x, self.dummy, op, L, C, 0, op["mean"], self.dummy)
                self.reduce(x, self.dummy, op, L, C, 1, self.dummy, self.dummy)
                k.bn_forward[grid(E), 256, self.stream](x, y, op["gamma"]["value"],
                    op["beta"]["value"], op["mean"], op["inv"], E, C)
            elif kind == "pool":
                k.pool_forward[grid(E), 256, self.stream](x, y, op["arg"], b, op["H"], op["W"], op["C"])
            else:
                k.dropout[grid(E), 256, self.stream](x, op["mask"], y, np.float32(1 / (1 - op["p"])), E)
            k.finite_check[grid(E), 256, self.stream](y, E, self.error)
        k.softmax_loss[b, 32, self.stream](self.ops[-1]["output"], self.labels,
            self.probs, self.g1, self.loss, self.correct, b)
        k.finite_check[grid(b), 256, self.stream](self.loss, b, self.error)

    def backward(self, b, audit_hook=None):
        current, other = self.g1, self.g2
        for index in range(len(self.ops) - 1, -1, -1):
            op = self.ops[index]
            kind, x = op["kind"], op["input"]
            E = b * op["sample_elements"]
            if kind in ("conv", "dense"):
                if op["relu"]:
                    k.relu_backward[grid(E), 256, self.stream](current, op["output"], E)
                if kind == "conv":
                    H, W, Ci, Co = (op[z] for z in ("H", "W", "Ci", "Co"))
                    L = b * H * W
                    target = op["weights"]["grad"] if self.backend == "cublas" else self.parts
                    self.gemm(2, x, current, self.dummy, target, 9 * Ci, Co, L, H, W, Ci, Co)
                    if self.backend == "explicit":
                        k.merge_partials[grid(9 * Ci * Co), 256, self.stream](self.parts,
                            op["weights"]["grad"], 9 * Ci * Co, (L + 2047) // 2048)
                    if index:
                        self.gemm(1, current, op["weights"]["value"], self.dummy, other,
                                  L, Ci, 9 * Co, H, W, Ci, Co)
                    C = Co
                else:
                    I, O, L, C = op["I"], op["O"], b, op["O"]
                    target = op["weights"]["grad"] if self.backend == "cublas" else self.parts
                    self.gemm(5, x, current, self.dummy, target, I, O, b, I=I, O=O)
                    if self.backend == "explicit":
                        k.merge_partials[grid(I * O), 256, self.stream](self.parts, op["weights"]["grad"], I * O, 1)
                    self.gemm(4, current, op["weights"]["value"], self.dummy, other, b, I, O, I=I, O=O)
                self.reduce(current, self.dummy, {}, L, C, 3, op["bias"]["grad"], self.dummy)
            elif kind == "bn":
                C, L = op["C"], E // op["C"]
                self.reduce(x, current, op, L, C, 2, op["beta"]["grad"], op["gamma"]["grad"])
                k.bn_backward[grid(E), 256, self.stream](x, current, other, op["gamma"]["value"],
                    op["mean"], op["inv"], op["beta"]["grad"], op["gamma"]["grad"], L, C)
            elif kind == "pool":
                E_in = b * op["H"] * op["W"] * op["C"]
                k.pool_backward[grid(E_in), 256, self.stream](current, op["arg"], other,
                    b, op["H"], op["W"], op["C"])
            else:
                k.dropout[grid(E), 256, self.stream](current, op["mask"], other,
                    np.float32(1 / (1 - op["p"])), E)
            if audit_hook is not None:
                self.stream.synchronize()
                audit_hook(index, op, current, other)
            current, other = other, current

    def update(self, b):
        t = np.float32(self.steps + 1)
        alpha = np.float32(.001) * np.sqrt(np.float32(1) - np.float32(.999)**t) / (
            np.float32(1) - np.float32(.9)**t)
        for p in self.params:
            k.adam[grid(p["value"].size), 256, self.stream](p["value"], p["grad"],
                p["m"], p["v"], np.float32(alpha), p["value"].size, self.error)
        k.metrics[1, 256, self.stream](self.loss, self.correct, b, self.total)
        k.advance_step[1, 1, self.stream](self.counter)
        self.steps += 1

    def train_step(self, x, labels, epoch=0, batch_index=0):
        self.upload(x, labels, epoch, batch_index)
        with self.timed("forward_loss"):
            self.forward(len(x))
        with self.timed("backward"):
            self.backward(len(x))
        with self.timed("adam_metricas"):
            self.update(len(x))
        with self.timed("d2h_control"):
            self.error.copy_to_host(self.host_error, stream=self.stream)
        times = self.drain_events()
        if self.host_error[0]:
            raise FloatingPointError(f"CUDA sticky error {self.host_error[0]}, step {self.steps}")
        return times

    def reset_metrics(self):
        self.total.copy_to_device(np.zeros(3, np.float32), stream=self.stream)

    def export(self, optimizer=False):
        """D2H model state, then install CPU mirror outside the transfer interval."""
        host = []
        with self.timed("d2h_modelo"):
            for p in self.params:
                host.append((p["variable"], p["value"].copy_to_host(stream=self.stream).reshape(p["initial"].shape)))
            for op in self.ops:
                if op["kind"] == "bn":
                    host.append((op["layer"].moving_mean, op["moving_mean"].copy_to_host(stream=self.stream)))
                    host.append((op["layer"].moving_variance, op["moving_var"].copy_to_host(stream=self.stream)))
            self.total.copy_to_host(self.host_total, stream=self.stream)
        times = self.drain_events()
        for variable, value in host:
            variable.assign(value)
        if optimizer:
            with self.timed("d2h_optimizador"):
                states = [(p["m"].copy_to_host(stream=self.stream), p["v"].copy_to_host(stream=self.stream)) for p in self.params]
                count = self.counter.copy_to_host(stream=self.stream)
            extra = self.drain_events()
            times.update(extra)
            if int(count[0]) != self.steps:
                raise ValueError("Device update counter differs")
            self.reference.optimizer.iterations.assign(int(count[0]))
            for i, (moment, velocity) in enumerate(states):
                self.reference.optimizer._momentums[i].assign(moment.reshape(self.params[i]["initial"].shape))
                self.reference.optimizer._velocities[i].assign(velocity.reshape(self.params[i]["initial"].shape))
        return times, self.host_total.copy()


In [ ]:
%%modulo_fc asl_cuda.fc
"""FC-A/FC-B CUDA extension. Frozen CNN-B code is reused, never edited.

All arithmetic training executes through explicit Numba CUDA launches and
cuBLAS FP32 SGEMM. CPU is restricted to preparation, validation and checkpoint I/O.
The BN finishing reduction uses one thread per input feature (12,288 features),
with momentum taken from the original FC layer, rather than CNN-B's 0.9.
"""
import hashlib
import json
import os
import sys
import numpy as np
from pathlib import Path
from .runtime import cuda, device_info
from .model import Network, grid, ROOT
from . import kernels as k
from numba import float32


@cuda.jit(cache=True, fastmath=False)
def finish_fc(p1,p2,out1,out2,movmean,movvar,mean,inv,P,C,L,mode,momentum):
    for c in range(cuda.grid(1),C,cuda.gridsize(1)):
        a,b=float32(0),float32(0)
        for p in range(P):
            a+=p1[p*C+c]
            b+=p2[p*C+c]
        if mode==0:
            out1[c]=a/float32(L)
        elif mode==1:
            var=a/float32(L)
            inv[c]=float32(1)/(var+float32(.001))**float32(.5)
            complement=float32(1)-momentum
            movmean[c]=momentum*movmean[c]+complement*mean[c]
            movvar[c]=momentum*movvar[c]+complement*var
        elif mode==2:
            out1[c],out2[c]=a,b
        else:
            out1[c]=a


def fc_reference(name):
    """Restore historical architecture, ALL variables and zero Adam state."""
    if name not in ('FC-A','FC-B'):raise ValueError(name)
    for key,value in {'TF_NUM_INTRAOP_THREADS':'6','TF_NUM_INTEROP_THREADS':'1',
        'OMP_NUM_THREADS':'6','OPENBLAS_NUM_THREADS':'6','MKL_NUM_THREADS':'6',
        'TF_DETERMINISTIC_OPS':'1','TF_ENABLE_ONEDNN_OPTS':'1'}.items():os.environ[key]=value
    import tensorflow as tf
    tf.config.set_visible_devices([], 'GPU')
    tf.config.threading.set_intra_op_parallelism_threads(6)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.keras.mixed_precision.set_global_policy('float32')
    seed=44 if name=='FC-A' else 45
    tf.keras.utils.set_random_seed(seed)
    sys.path.insert(0,str(ROOT/'scripts'))
    from benchmark_cpu import original_namespace
    ns=original_namespace();ns.update(keras=tf.keras,layers=tf.keras.layers,NUM_CLASSES=29)
    model=ns['build_fc']([128,64] if name=='FC-A' else [512,256,128],
        dropout=.3 if name=='FC-A' else .4,name=name)
    model.jit_compile=False
    model.optimizer.build(model.trainable_variables)
    initial=ROOT/'data/EXP-002/RUN-20261117T171942Z-cpu-01'/f'{name}.estado-inicial.npz'
    index=json.loads((ROOT/'resultados/secuencial/EXP-002/RUN-20261117T171942Z-cpu-01/estados-iniciales.json').read_text(encoding='utf8'))
    # Both the checkpoint bytes and its source metadata are frozen by EXP-010.
    with np.load(initial,allow_pickle=False) as data:
        variables=[*model.variables,*model.optimizer.variables]
        if len(variables)!=len(data.files):raise ValueError('Initial variable count')
        for i,v in enumerate(variables):
            a=data[f'arr_{i}']
            if tuple(v.shape)!=a.shape or str(v.dtype)!=str(a.dtype):raise ValueError('Initial shape/dtype')
            v.assign(a)
    if int(model.optimizer.iterations.numpy())!=0:raise ValueError('Nonzero initial step')
    if any(np.any(v.numpy()) for v in [*model.optimizer._momentums,*model.optimizer._velocities]):raise ValueError('Nonzero Adam')
    return tf,model,seed,initial


class FCNetwork(Network):
    def __init__(self, reference, seed):
        backend = "cublas"
        self.seed = int(seed)
        self.info = device_info()
        if self.info["free_bytes"] < 512 * 2**20:
            raise MemoryError("Insufficient free VRAM for the frozen profile")
        self.stream = cuda.stream()
        if backend not in ("explicit", "cublas"):
            raise ValueError("Unknown backend")
        self.backend = backend
        self.allocations, self.params, self.ops, self.transfers = [], [], [], []
        self.reference = reference
        self.input = self.alloc(128 * 64 * 64 * 3)
        self.host_x = cuda.pinned_array(128 * 64 * 64 * 3, dtype=np.float32)
        self.labels = self.alloc(128, np.int64)
        self.host_y = cuda.pinned_array(128, dtype=np.int64)
        shape, previous = (64, 64, 3), self.input
        for layer in reference.layers:
            name = type(layer).__name__
            item = dict(input=previous, shape=shape, layer=layer)
            if name == "Conv2D":
                H, W, Ci = shape
                Co = layer.filters
                item.update(kind="conv", H=H, W=W, Ci=Ci, Co=Co,
                            weights=self.param(layer.kernel), bias=self.param(layer.bias), relu=True)
                shape = (H, W, Co)
            elif name == "Dense":
                I, O = int(layer.kernel.shape[0]), int(layer.kernel.shape[1])
                item.update(kind="dense", I=I, O=O, relu=O != 29,
                            weights=self.param(layer.kernel), bias=self.param(layer.bias))
                shape = (O,)
            elif name == "BatchNormalization":
                C = shape[-1]
                item.update(kind="bn", C=C, momentum=float(layer.momentum), gamma=self.param(layer.gamma), beta=self.param(layer.beta),
                            moving_mean=self.alloc(C), moving_var=self.alloc(C),
                            initial_mean=layer.moving_mean.numpy().copy(), initial_var=layer.moving_variance.numpy().copy(),
                            mean=self.alloc(C), inv=self.alloc(C))
            elif name == "MaxPooling2D":
                H, W, C = shape
                item.update(kind="pool", H=H, W=W, C=C)
                shape = (H // 2, W // 2, C)
                item["arg"] = self.alloc(128 * int(np.prod(shape)), np.uint8)
            elif name == "Dropout":
                E = 128 * int(np.prod(shape))
                item.update(kind="drop", p=layer.rate, mask=self.alloc(E, np.uint8),
                            host_mask=cuda.pinned_array(E, dtype=np.uint8))
            elif name == "Flatten":
                shape = (int(np.prod(shape)),)
                continue
            elif name == "InputLayer":
                continue
            else:
                raise TypeError(f"Unsupported original layer {name}")
            item["sample_elements"] = int(np.prod(shape))
            item["output"] = self.alloc(128 * item["sample_elements"])
            self.ops.append(item)
            previous = item["output"]
        self.probs = self.alloc(128 * 29)
        self.loss, self.correct = self.alloc(128), self.alloc(128, np.uint8)
        self.g1, self.g2 = self.alloc(max(128 * 12288, max(128 * op["sample_elements"] for op in self.ops))), self.alloc(max(128 * 12288, max(128 * op["sample_elements"] for op in self.ops)))
        self.parts = self.alloc(256)
        self.p1, self.p2 = self.alloc(65536), self.alloc(65536)
        self.dummy = self.alloc(256)
        self.total = self.alloc(3)
        self.error = self.alloc(1, np.int32)
        self.counter = self.alloc(1, np.int64)
        self.host_error = cuda.pinned_array(1, dtype=np.int32)
        self.host_total = cuda.pinned_array(3, dtype=np.float32)
        if backend == "cublas":
            from .blas import Blas
            self.blas = Blas(self.stream)
            self.columns = None  # FC has no convolutions or im2col workspace
        self.bytes_allocated = sum(a.nbytes for a in self.allocations)
        if self.bytes_allocated > 384 * 2**20:
            raise MemoryError(f"Arena exceeds design: {self.bytes_allocated}")
        if sum(p["value"].size for p in self.params) != sum(int(np.prod(v.shape)) for v in reference.trainable_variables):
            raise ValueError("Architecture parameter count changed")
        if any(op["kind"] not in ("dense", "bn", "drop") for op in self.ops):
            raise TypeError("FC profile only")
        self.events = []
        self.restore()


    def upload(self, x, labels, epoch, batch_index):
        b = len(x)
        np.copyto(self.host_x[:x.size], x.ravel())
        np.copyto(self.host_y[:b], labels)
        drops = [op for op in self.ops if op["kind"] == "drop"]
        for d, op in enumerate(drops):
            E = b * op["sample_elements"]
            rng = np.random.Generator(np.random.PCG64(np.random.SeedSequence([self.seed, epoch, batch_index, d + 1])))
            op["host_mask"][:E] = rng.random(E, dtype=np.float32) >= np.float32(op["p"])
        with self.timed("h2d_lotes"):
            self.input[:x.size].copy_to_device(self.host_x[:x.size], stream=self.stream)
            self.labels[:b].copy_to_device(self.host_y[:b], stream=self.stream)
            for op in drops:
                E = b * op["sample_elements"]
                op["mask"][:E].copy_to_device(op["host_mask"][:E], stream=self.stream)
        self.transfers.append(dict(batch=b, imagenes_bytes=x.nbytes, etiquetas_bytes=labels.nbytes,
                                   mascaras_bytes=sum(b * op["sample_elements"] for op in drops)))

    def reduce(self,x,dy,op,L,C,mode,out1,out2):
        P=(L+2047)//2048
        if P*C>self.p1.size:raise ValueError('FC reduction workspace')
        mean,inv=op.get('mean',self.dummy),op.get('inv',self.dummy)
        k.reduce_channels[(P,(C+31)//32),(32,8),self.stream](
            x,dy,mean,inv,self.p1,self.p2,L,C,2 if mode==2 else (1 if mode==1 else 0))
        finish_fc[grid(C),256,self.stream](self.p1,self.p2,out1,out2,
            op.get('moving_mean',self.dummy),op.get('moving_var',self.dummy),
            mean,inv,P,C,L,mode,np.float32(op.get('momentum',.99)))


In [ ]:
from asl_cuda.fc import FCNetwork,fc_reference,finish_fc
from asl_cuda.model import compiled_resources
from asl_cuda.runtime import cuda
expected=json.loads((ROOT/'resultados/secuencial/EXP-002/RUN-20261117T171942Z-cpu-01/tensores.json').read_text(encoding='utf8'))
all_results=[dict(run_id='RUN-20261118T134323Z-fc-a-01',**json.loads((ROOT/'resultados/paralelo/EXP-011/RUN-20261118T134323Z-fc-a-01/metricas.json').read_text(encoding='utf8')))]
def state(phase,**other):
    write_json(run/'estado.json',dict(utc=utc(),fase=phase,modelo=name,**other))
def add(phase,seconds,kind='pared',**other):
    timings.append(dict(experimento_id='EXP-012',run_id=run.name,modelo=name,repeticion=repetition,
        fase=phase,tipo_medicion=kind,duracion_s=float(seconds),configuracion_sha256=sha(CONFIG),**dict({'inicio_utc':'','fin_utc':''},**other)))
def total_events(extra):
    for key,value in extra.items():events[key]=events.get(key,0.)+float(value)


## FC-A — cinco ejecuciones nuevas

Batch128, último lote20, 15épocas, 1440actualizaciones. La medida pared incluye transferencias y validación CPU; guardado y gráficas quedan fuera. La primera corrida incorpora captura en vivo; su coste no se resta.

### FC-A: corrida 2

RUN: `RUN-20261118T134514Z-fc-a-02`. Inicialización/restauración → preparación CPU → entrenamiento CUDA → evaluación CPU y evidencias.

In [ ]:
name='FC-A';repetition=2;run_id='RUN-20261118T134514Z-fc-a-02'
run=ROOT/'resultados/paralelo/EXP-012'/run_id
run.mkdir(exist_ok=False)
timings=[];epoch_rows=[];events={};shuffle_hashes=[]
state('inicializacion')
stop,monitor=iniciar_monitor(run)
if repetition==1:
    capture_log=(run/'capturador.log').open('x',encoding='utf8')
    capture=subprocess.Popen([sys.executable,'-X','utf8',str(ROOT/'scripts/capturar_fc_real.py'),
        '--run',str(run),'--tipo','cuda','--modelo',name,'--nombre',name.lower()],
        stdout=capture_log,stderr=subprocess.STDOUT)
init_start=time.perf_counter()
tf,reference,seed,initial=fc_reference(name)
assert sha(initial)==config['estados_iniciales'][name]['sha256']
net=FCNetwork(reference,seed)
reference.summary()
print('GPU REAL',net.info,'Arena bytes',net.bytes_allocated,'BN momentum',[op['momentum'] for op in net.ops if op['kind']=='bn'])
write_json(run/'hardware.json',dict(device=net.info,tensorflow=tf.__version__,
    tf_build=tf.sysconfig.get_build_info(),cpu_tensor_device=tf.constant([1.]).device,
    tf_visible=[str(d) for d in tf.config.get_visible_devices()],cublas_version=net.blas.version,
    cublas_dll=str(net.blas.path),cublas_sha256=sha(net.blas.path),buffers_bytes=net.bytes_allocated,
    parametros=sum(p['value'].size for p in net.params),initial_checkpoint_sha256=sha(initial)))
assert not tf.config.get_visible_devices('GPU')
wx,wy=load_images(manifest['train'][:148],data_root,1)
wv,wvy=load_images(manifest['val'][:128],data_root,1)
wt,wty=load_images(manifest['test'][:32],data_root,1)
net.train_step(wx[:128],wy[:128],0,0);net.train_step(wx[128:],wy[128:],0,1)
reference.evaluate(wv,wvy,batch_size=128,verbose=0)
reference.evaluate(wt,wty,batch_size=32,verbose=0)
net.restore()
for p in net.params:
    assert np.array_equal(p['value'].copy_to_host(),p['initial'].ravel())
    assert not np.any(p['m'].copy_to_host()) and not np.any(p['v'].copy_to_host())
for op in net.ops:
    if op['kind']=='bn':
        assert np.array_equal(op['moving_mean'].copy_to_host(),op['initial_mean'])
        assert np.array_equal(op['moving_var'].copy_to_host(),op['initial_var'])
assert net.counter.copy_to_host()[0]==0
write_json(run/'warmup.json',dict(duracion_s=time.perf_counter()-init_start,train=148,val=128,test=32,
    restauracion_verificada=True,excluido_timing=True))
print('Warmup nuevo FC terminado y estado inicial restaurado; no incluido en entrenamiento.')


In [ ]:
state('preparacion_cpu',workers=6)
start=time.perf_counter()
inputs={split:load_images(manifest[split],data_root,6) for split in ['train','val','test']}
add('preparacion_cpu',time.perf_counter()-start)
tensors={}
for split,(images,labels) in inputs.items():
    a=hashlib.sha256(memoryview(images).cast('B')).hexdigest()
    b=hashlib.sha256(memoryview(labels).cast('B')).hexdigest()
    assert a==expected[split]['images_sha256'] and b==expected[split]['labels_sha256']
    tensors[split]=dict(images_sha256=a,labels_sha256=b,shape=list(images.shape),dtype=str(images.dtype))
write_json(run/'tensores.json',tensors)
x,y=inputs['train'];vx,vy=inputs['val'];tx,ty=inputs['test']
print('Preparación SOLO CPU (6 workers); bytes idénticos a EXP-002:',tensors)


In [ ]:
state('entrenamiento_'+name,epoca=1,pasos=0)
train_start_utc=utc();train_start=time.perf_counter_ns()
validations=0.;export_wall=0.
with net.timed('h2d_inicial'):net.restore()
total_events(net.drain_events())
rng=np.random.Generator(np.random.PCG64(seed))
for epoch in range(15):
    epoch_start=time.perf_counter()
    state('entrenamiento_'+name,epoca=epoch+1,pasos=net.steps)
    net.reset_metrics()
    order=rng.permutation(len(x))
    shuffle_hashes.append(hashlib.sha256(order.astype(np.int64).tobytes()).hexdigest())
    for j,begin in enumerate(range(0,len(x),128)):
        indices=order[begin:begin+128]
        total_events(net.train_step(x[indices],y[indices],epoch,j))
    export_start=time.perf_counter()
    extra,total=net.export();total_events(extra)
    export_wall+=time.perf_counter()-export_start
    assert int(total[2])==12180
    state('validacion_cpu',epoca=epoch+1,pasos=net.steps)
    val_start=time.perf_counter()
    val=reference.evaluate(vx,vy,batch_size=128,verbose=0,return_dict=True)
    val_s=time.perf_counter()-val_start;validations+=val_s
    epoch_rows.append(dict(modelo=name,repeticion=repetition,epoca=epoch+1,pasos_acumulados=net.steps,
        train_loss=float(total[0]/total[2]),train_accuracy=float(total[1]/total[2]),
        val_loss=float(val['loss']),val_accuracy=float(val['accuracy']),
        pared_epoca_s=time.perf_counter()-epoch_start,validacion_cpu_s=val_s))
    write_csv(run/'epocas.csv',epoch_rows)
    print('EPOCH',name,repetition,epoch_rows[-1],flush=True)
extra,_=net.export(optimizer=True);total_events(extra)
train_seconds=(time.perf_counter_ns()-train_start)/1e9
train_end_utc=utc()
assert net.steps==1440 and int(reference.optimizer.iterations.numpy())==1440
add('entrenamiento_'+name,train_seconds,inicio_utc=train_start_utc,fin_utc=train_end_utc)
add('validacion_anidada_'+name,validations)
for phase,seconds in events.items():add(phase,seconds,'evento_device')
print('ENTRENAMIENTO CUDA REAL',name,'corrida',repetition,'segundos',train_seconds,
    '| incluye staging, máscaras, H2D, GPU, D2H y validación CPU; pasos',net.steps)


In [ ]:
state('evaluacion_final_cpu',pasos=net.steps)
start=time.perf_counter_ns()
test=reference.evaluate(tx,ty,batch_size=32,verbose=0,return_dict=True)
eval_seconds=(time.perf_counter_ns()-start)/1e9
add('evaluacion_'+name,eval_seconds)
metrics=dict(modelo=name,repeticion=repetition,test={k:float(v) for k,v in test.items()},epocas=15,pasos=1440,
    entrenamiento_s=train_seconds,evaluacion_cpu_s=eval_seconds,
    comparabilidad='Datos, estado inicial, arquitectura y pasos equivalentes; CPU histórica; RNG interno diferente')
write_json(run/'metricas.json',metrics);write_csv(run/'tiempos.csv',timings)
write_json(run/'eventos.json',dict(eventos_s=events,validacion_cpu_s=validations,export_cpu_pared_s=export_wall,
    nota='Intervalos stream anidados; no equivalen a tiempo activo exclusivo de kernels; no sumar al tiempo pared'))
write_json(run/'ordenes.json',dict(seed=seed,generador='PCG64',epocas_sha256=shuffle_hashes))
param_count=sum(p['value'].size for p in net.params)
bn_count=sum(op['C']*2 for op in net.ops if op['kind']=='bn')
write_json(run/'transferencias.json',dict(lotes=net.transfers[-1440:],modelo_d2h_copias=16,
    modelo_bytes_por_copia=(param_count+bn_count)*4,optimizador_final_bytes=param_count*8+8,
    control_d2h_bytes=1440*4,reinicios_metricas_h2d_bytes=15*12,
    nota='Copia final adicional de pesos incluida; sin restar costes'))
state('guardado_fuera_timing')
model_dir=ROOT/'data/EXP-012'/run.name;model_dir.mkdir(parents=True,exist_ok=False)
model_path=model_dir/(name+'-CUDA.keras');reference.save(model_path)
fingerprint=hashlib.sha256()
for variable in [*reference.variables,*reference.optimizer.variables]:
    array=variable.numpy();fingerprint.update(str(array.dtype).encode());fingerprint.update(str(array.shape).encode());fingerprint.update(array.tobytes())
write_json(run/'modelo-guardado.json',dict(ruta=model_path.relative_to(ROOT).as_posix(),sha256=sha(model_path),
    estado_canonico_sha256=fingerprint.hexdigest(),optimizer_iterations=1440))
write_json(run/'lanzamientos.json',dict(vector_block=256,bn_parcial_block=[32,8,1],
    bn_parcial_grid=[1,384,1],bn_final_block=256,bn_final_grid=[48,1,1],
    softmax_block=32,softmax_grid=[128,1,1],ultimo_batch=20,recursos=compiled_resources(),
    fc_finish_regs=str(finish_fc.get_regs_per_thread()),
    gemm='cuBLAS SGEMM FP32 PEDANTIC; jerarquía interna de biblioteca, no fijada por el equipo'))
state('completada',pasos=1440)
stop.set();monitor.join(timeout=15)
if repetition==1:
    try:capture.wait(timeout=8)
    except subprocess.TimeoutExpired:capture.terminate();capture.wait();print('Captura no disponible; revisar capturador.log')
    capture_log.close()
display(pd.DataFrame(timings).fillna(''))
display(pd.DataFrame(epoch_rows))
print('MÉTRICAS REALES',json.dumps(metrics,indent=2,ensure_ascii=False))
fig,axes=plt.subplots(1,2,figsize=(10,3.6),layout='constrained')
for ax,keys,title in zip(axes,[('train_loss','val_loss'),('train_accuracy','val_accuracy')],['Pérdida','Exactitud']):
    for key in keys:ax.plot(range(1,16),[r[key] for r in epoch_rows],label=key)
    ax.set(xlabel='Época',ylabel=title,title=name+' · corrida '+str(repetition));ax.legend();ax.grid(alpha=.2)
fig.savefig(run/'curvas.png',dpi=160);fig.savefig(run/'curvas.svg');mostrar_figura(fig)
resource=pd.read_csv(run/'recursos.csv')
fig,axes=plt.subplots(2,1,figsize=(10,5),sharex=True,layout='constrained')
axes[0].plot(resource.transcurrido_s,resource.cpu_procesos_pct,color='#0072B2',label='CPU proceso e hijos (100% por núcleo)')
axes[1].plot(resource.transcurrido_s,resource.gpu_util_pct,color='#D55E00',label='Actividad GPU (%)')
axes[0].set(ylabel='Uso CPU (%)');axes[1].set(ylabel='Actividad GPU (%)',xlabel='Tiempo desde inicio del RUN (s)',ylim=(0,105))
for ax in axes:ax.legend(loc='upper right');ax.grid(alpha=.2)
fig.suptitle(name+' · monitorización real, corrida '+str(repetition))
fig.savefig(run/'recursos.png',dpi=160);fig.savefig(run/'recursos.svg');mostrar_figura(fig)
for picture in sorted(run.glob('captura-*.png')):display(Image(filename=str(picture)))
all_results.append(dict(run_id=run.name,**metrics));write_json(ROOT/'resultados/paralelo/EXP-012/progreso.json',all_results)
net.stream.synchronize();net.blas.close()
del net,reference,inputs,x,y,vx,vy,tx,ty,wx,wy,wv,wvy,wt,wty,p,op
tf.keras.backend.clear_session();gc.collect();cuda.current_context().deallocations.clear()


### FC-A: corrida 3

RUN: `RUN-20261118T134514Z-fc-a-03`. Inicialización/restauración → preparación CPU → entrenamiento CUDA → evaluación CPU y evidencias.

In [ ]:
name='FC-A';repetition=3;run_id='RUN-20261118T134514Z-fc-a-03'
run=ROOT/'resultados/paralelo/EXP-012'/run_id
run.mkdir(exist_ok=False)
timings=[];epoch_rows=[];events={};shuffle_hashes=[]
state('inicializacion')
stop,monitor=iniciar_monitor(run)
if repetition==1:
    capture_log=(run/'capturador.log').open('x',encoding='utf8')
    capture=subprocess.Popen([sys.executable,'-X','utf8',str(ROOT/'scripts/capturar_fc_real.py'),
        '--run',str(run),'--tipo','cuda','--modelo',name,'--nombre',name.lower()],
        stdout=capture_log,stderr=subprocess.STDOUT)
init_start=time.perf_counter()
tf,reference,seed,initial=fc_reference(name)
assert sha(initial)==config['estados_iniciales'][name]['sha256']
net=FCNetwork(reference,seed)
reference.summary()
print('GPU REAL',net.info,'Arena bytes',net.bytes_allocated,'BN momentum',[op['momentum'] for op in net.ops if op['kind']=='bn'])
write_json(run/'hardware.json',dict(device=net.info,tensorflow=tf.__version__,
    tf_build=tf.sysconfig.get_build_info(),cpu_tensor_device=tf.constant([1.]).device,
    tf_visible=[str(d) for d in tf.config.get_visible_devices()],cublas_version=net.blas.version,
    cublas_dll=str(net.blas.path),cublas_sha256=sha(net.blas.path),buffers_bytes=net.bytes_allocated,
    parametros=sum(p['value'].size for p in net.params),initial_checkpoint_sha256=sha(initial)))
assert not tf.config.get_visible_devices('GPU')
wx,wy=load_images(manifest['train'][:148],data_root,1)
wv,wvy=load_images(manifest['val'][:128],data_root,1)
wt,wty=load_images(manifest['test'][:32],data_root,1)
net.train_step(wx[:128],wy[:128],0,0);net.train_step(wx[128:],wy[128:],0,1)
reference.evaluate(wv,wvy,batch_size=128,verbose=0)
reference.evaluate(wt,wty,batch_size=32,verbose=0)
net.restore()
for p in net.params:
    assert np.array_equal(p['value'].copy_to_host(),p['initial'].ravel())
    assert not np.any(p['m'].copy_to_host()) and not np.any(p['v'].copy_to_host())
for op in net.ops:
    if op['kind']=='bn':
        assert np.array_equal(op['moving_mean'].copy_to_host(),op['initial_mean'])
        assert np.array_equal(op['moving_var'].copy_to_host(),op['initial_var'])
assert net.counter.copy_to_host()[0]==0
write_json(run/'warmup.json',dict(duracion_s=time.perf_counter()-init_start,train=148,val=128,test=32,
    restauracion_verificada=True,excluido_timing=True))
print('Warmup nuevo FC terminado y estado inicial restaurado; no incluido en entrenamiento.')


In [ ]:
state('preparacion_cpu',workers=6)
start=time.perf_counter()
inputs={split:load_images(manifest[split],data_root,6) for split in ['train','val','test']}
add('preparacion_cpu',time.perf_counter()-start)
tensors={}
for split,(images,labels) in inputs.items():
    a=hashlib.sha256(memoryview(images).cast('B')).hexdigest()
    b=hashlib.sha256(memoryview(labels).cast('B')).hexdigest()
    assert a==expected[split]['images_sha256'] and b==expected[split]['labels_sha256']
    tensors[split]=dict(images_sha256=a,labels_sha256=b,shape=list(images.shape),dtype=str(images.dtype))
write_json(run/'tensores.json',tensors)
x,y=inputs['train'];vx,vy=inputs['val'];tx,ty=inputs['test']
print('Preparación SOLO CPU (6 workers); bytes idénticos a EXP-002:',tensors)


In [ ]:
state('entrenamiento_'+name,epoca=1,pasos=0)
train_start_utc=utc();train_start=time.perf_counter_ns()
validations=0.;export_wall=0.
with net.timed('h2d_inicial'):net.restore()
total_events(net.drain_events())
rng=np.random.Generator(np.random.PCG64(seed))
for epoch in range(15):
    epoch_start=time.perf_counter()
    state('entrenamiento_'+name,epoca=epoch+1,pasos=net.steps)
    net.reset_metrics()
    order=rng.permutation(len(x))
    shuffle_hashes.append(hashlib.sha256(order.astype(np.int64).tobytes()).hexdigest())
    for j,begin in enumerate(range(0,len(x),128)):
        indices=order[begin:begin+128]
        total_events(net.train_step(x[indices],y[indices],epoch,j))
    export_start=time.perf_counter()
    extra,total=net.export();total_events(extra)
    export_wall+=time.perf_counter()-export_start
    assert int(total[2])==12180
    state('validacion_cpu',epoca=epoch+1,pasos=net.steps)
    val_start=time.perf_counter()
    val=reference.evaluate(vx,vy,batch_size=128,verbose=0,return_dict=True)
    val_s=time.perf_counter()-val_start;validations+=val_s
    epoch_rows.append(dict(modelo=name,repeticion=repetition,epoca=epoch+1,pasos_acumulados=net.steps,
        train_loss=float(total[0]/total[2]),train_accuracy=float(total[1]/total[2]),
        val_loss=float(val['loss']),val_accuracy=float(val['accuracy']),
        pared_epoca_s=time.perf_counter()-epoch_start,validacion_cpu_s=val_s))
    write_csv(run/'epocas.csv',epoch_rows)
    print('EPOCH',name,repetition,epoch_rows[-1],flush=True)
extra,_=net.export(optimizer=True);total_events(extra)
train_seconds=(time.perf_counter_ns()-train_start)/1e9
train_end_utc=utc()
assert net.steps==1440 and int(reference.optimizer.iterations.numpy())==1440
add('entrenamiento_'+name,train_seconds,inicio_utc=train_start_utc,fin_utc=train_end_utc)
add('validacion_anidada_'+name,validations)
for phase,seconds in events.items():add(phase,seconds,'evento_device')
print('ENTRENAMIENTO CUDA REAL',name,'corrida',repetition,'segundos',train_seconds,
    '| incluye staging, máscaras, H2D, GPU, D2H y validación CPU; pasos',net.steps)


In [ ]:
state('evaluacion_final_cpu',pasos=net.steps)
start=time.perf_counter_ns()
test=reference.evaluate(tx,ty,batch_size=32,verbose=0,return_dict=True)
eval_seconds=(time.perf_counter_ns()-start)/1e9
add('evaluacion_'+name,eval_seconds)
metrics=dict(modelo=name,repeticion=repetition,test={k:float(v) for k,v in test.items()},epocas=15,pasos=1440,
    entrenamiento_s=train_seconds,evaluacion_cpu_s=eval_seconds,
    comparabilidad='Datos, estado inicial, arquitectura y pasos equivalentes; CPU histórica; RNG interno diferente')
write_json(run/'metricas.json',metrics);write_csv(run/'tiempos.csv',timings)
write_json(run/'eventos.json',dict(eventos_s=events,validacion_cpu_s=validations,export_cpu_pared_s=export_wall,
    nota='Intervalos stream anidados; no equivalen a tiempo activo exclusivo de kernels; no sumar al tiempo pared'))
write_json(run/'ordenes.json',dict(seed=seed,generador='PCG64',epocas_sha256=shuffle_hashes))
param_count=sum(p['value'].size for p in net.params)
bn_count=sum(op['C']*2 for op in net.ops if op['kind']=='bn')
write_json(run/'transferencias.json',dict(lotes=net.transfers[-1440:],modelo_d2h_copias=16,
    modelo_bytes_por_copia=(param_count+bn_count)*4,optimizador_final_bytes=param_count*8+8,
    control_d2h_bytes=1440*4,reinicios_metricas_h2d_bytes=15*12,
    nota='Copia final adicional de pesos incluida; sin restar costes'))
state('guardado_fuera_timing')
model_dir=ROOT/'data/EXP-012'/run.name;model_dir.mkdir(parents=True,exist_ok=False)
model_path=model_dir/(name+'-CUDA.keras');reference.save(model_path)
fingerprint=hashlib.sha256()
for variable in [*reference.variables,*reference.optimizer.variables]:
    array=variable.numpy();fingerprint.update(str(array.dtype).encode());fingerprint.update(str(array.shape).encode());fingerprint.update(array.tobytes())
write_json(run/'modelo-guardado.json',dict(ruta=model_path.relative_to(ROOT).as_posix(),sha256=sha(model_path),
    estado_canonico_sha256=fingerprint.hexdigest(),optimizer_iterations=1440))
write_json(run/'lanzamientos.json',dict(vector_block=256,bn_parcial_block=[32,8,1],
    bn_parcial_grid=[1,384,1],bn_final_block=256,bn_final_grid=[48,1,1],
    softmax_block=32,softmax_grid=[128,1,1],ultimo_batch=20,recursos=compiled_resources(),
    fc_finish_regs=str(finish_fc.get_regs_per_thread()),
    gemm='cuBLAS SGEMM FP32 PEDANTIC; jerarquía interna de biblioteca, no fijada por el equipo'))
state('completada',pasos=1440)
stop.set();monitor.join(timeout=15)
if repetition==1:
    try:capture.wait(timeout=8)
    except subprocess.TimeoutExpired:capture.terminate();capture.wait();print('Captura no disponible; revisar capturador.log')
    capture_log.close()
display(pd.DataFrame(timings).fillna(''))
display(pd.DataFrame(epoch_rows))
print('MÉTRICAS REALES',json.dumps(metrics,indent=2,ensure_ascii=False))
fig,axes=plt.subplots(1,2,figsize=(10,3.6),layout='constrained')
for ax,keys,title in zip(axes,[('train_loss','val_loss'),('train_accuracy','val_accuracy')],['Pérdida','Exactitud']):
    for key in keys:ax.plot(range(1,16),[r[key] for r in epoch_rows],label=key)
    ax.set(xlabel='Época',ylabel=title,title=name+' · corrida '+str(repetition));ax.legend();ax.grid(alpha=.2)
fig.savefig(run/'curvas.png',dpi=160);fig.savefig(run/'curvas.svg');mostrar_figura(fig)
resource=pd.read_csv(run/'recursos.csv')
fig,axes=plt.subplots(2,1,figsize=(10,5),sharex=True,layout='constrained')
axes[0].plot(resource.transcurrido_s,resource.cpu_procesos_pct,color='#0072B2',label='CPU proceso e hijos (100% por núcleo)')
axes[1].plot(resource.transcurrido_s,resource.gpu_util_pct,color='#D55E00',label='Actividad GPU (%)')
axes[0].set(ylabel='Uso CPU (%)');axes[1].set(ylabel='Actividad GPU (%)',xlabel='Tiempo desde inicio del RUN (s)',ylim=(0,105))
for ax in axes:ax.legend(loc='upper right');ax.grid(alpha=.2)
fig.suptitle(name+' · monitorización real, corrida '+str(repetition))
fig.savefig(run/'recursos.png',dpi=160);fig.savefig(run/'recursos.svg');mostrar_figura(fig)
for picture in sorted(run.glob('captura-*.png')):display(Image(filename=str(picture)))
all_results.append(dict(run_id=run.name,**metrics));write_json(ROOT/'resultados/paralelo/EXP-012/progreso.json',all_results)
net.stream.synchronize();net.blas.close()
del net,reference,inputs,x,y,vx,vy,tx,ty,wx,wy,wv,wvy,wt,wty,p,op
tf.keras.backend.clear_session();gc.collect();cuda.current_context().deallocations.clear()


### FC-A: corrida 4

RUN: `RUN-20261118T134514Z-fc-a-04`. Inicialización/restauración → preparación CPU → entrenamiento CUDA → evaluación CPU y evidencias.

In [ ]:
name='FC-A';repetition=4;run_id='RUN-20261118T134514Z-fc-a-04'
run=ROOT/'resultados/paralelo/EXP-012'/run_id
run.mkdir(exist_ok=False)
timings=[];epoch_rows=[];events={};shuffle_hashes=[]
state('inicializacion')
stop,monitor=iniciar_monitor(run)
if repetition==1:
    capture_log=(run/'capturador.log').open('x',encoding='utf8')
    capture=subprocess.Popen([sys.executable,'-X','utf8',str(ROOT/'scripts/capturar_fc_real.py'),
        '--run',str(run),'--tipo','cuda','--modelo',name,'--nombre',name.lower()],
        stdout=capture_log,stderr=subprocess.STDOUT)
init_start=time.perf_counter()
tf,reference,seed,initial=fc_reference(name)
assert sha(initial)==config['estados_iniciales'][name]['sha256']
net=FCNetwork(reference,seed)
reference.summary()
print('GPU REAL',net.info,'Arena bytes',net.bytes_allocated,'BN momentum',[op['momentum'] for op in net.ops if op['kind']=='bn'])
write_json(run/'hardware.json',dict(device=net.info,tensorflow=tf.__version__,
    tf_build=tf.sysconfig.get_build_info(),cpu_tensor_device=tf.constant([1.]).device,
    tf_visible=[str(d) for d in tf.config.get_visible_devices()],cublas_version=net.blas.version,
    cublas_dll=str(net.blas.path),cublas_sha256=sha(net.blas.path),buffers_bytes=net.bytes_allocated,
    parametros=sum(p['value'].size for p in net.params),initial_checkpoint_sha256=sha(initial)))
assert not tf.config.get_visible_devices('GPU')
wx,wy=load_images(manifest['train'][:148],data_root,1)
wv,wvy=load_images(manifest['val'][:128],data_root,1)
wt,wty=load_images(manifest['test'][:32],data_root,1)
net.train_step(wx[:128],wy[:128],0,0);net.train_step(wx[128:],wy[128:],0,1)
reference.evaluate(wv,wvy,batch_size=128,verbose=0)
reference.evaluate(wt,wty,batch_size=32,verbose=0)
net.restore()
for p in net.params:
    assert np.array_equal(p['value'].copy_to_host(),p['initial'].ravel())
    assert not np.any(p['m'].copy_to_host()) and not np.any(p['v'].copy_to_host())
for op in net.ops:
    if op['kind']=='bn':
        assert np.array_equal(op['moving_mean'].copy_to_host(),op['initial_mean'])
        assert np.array_equal(op['moving_var'].copy_to_host(),op['initial_var'])
assert net.counter.copy_to_host()[0]==0
write_json(run/'warmup.json',dict(duracion_s=time.perf_counter()-init_start,train=148,val=128,test=32,
    restauracion_verificada=True,excluido_timing=True))
print('Warmup nuevo FC terminado y estado inicial restaurado; no incluido en entrenamiento.')


In [ ]:
state('preparacion_cpu',workers=6)
start=time.perf_counter()
inputs={split:load_images(manifest[split],data_root,6) for split in ['train','val','test']}
add('preparacion_cpu',time.perf_counter()-start)
tensors={}
for split,(images,labels) in inputs.items():
    a=hashlib.sha256(memoryview(images).cast('B')).hexdigest()
    b=hashlib.sha256(memoryview(labels).cast('B')).hexdigest()
    assert a==expected[split]['images_sha256'] and b==expected[split]['labels_sha256']
    tensors[split]=dict(images_sha256=a,labels_sha256=b,shape=list(images.shape),dtype=str(images.dtype))
write_json(run/'tensores.json',tensors)
x,y=inputs['train'];vx,vy=inputs['val'];tx,ty=inputs['test']
print('Preparación SOLO CPU (6 workers); bytes idénticos a EXP-002:',tensors)


In [ ]:
state('entrenamiento_'+name,epoca=1,pasos=0)
train_start_utc=utc();train_start=time.perf_counter_ns()
validations=0.;export_wall=0.
with net.timed('h2d_inicial'):net.restore()
total_events(net.drain_events())
rng=np.random.Generator(np.random.PCG64(seed))
for epoch in range(15):
    epoch_start=time.perf_counter()
    state('entrenamiento_'+name,epoca=epoch+1,pasos=net.steps)
    net.reset_metrics()
    order=rng.permutation(len(x))
    shuffle_hashes.append(hashlib.sha256(order.astype(np.int64).tobytes()).hexdigest())
    for j,begin in enumerate(range(0,len(x),128)):
        indices=order[begin:begin+128]
        total_events(net.train_step(x[indices],y[indices],epoch,j))
    export_start=time.perf_counter()
    extra,total=net.export();total_events(extra)
    export_wall+=time.perf_counter()-export_start
    assert int(total[2])==12180
    state('validacion_cpu',epoca=epoch+1,pasos=net.steps)
    val_start=time.perf_counter()
    val=reference.evaluate(vx,vy,batch_size=128,verbose=0,return_dict=True)
    val_s=time.perf_counter()-val_start;validations+=val_s
    epoch_rows.append(dict(modelo=name,repeticion=repetition,epoca=epoch+1,pasos_acumulados=net.steps,
        train_loss=float(total[0]/total[2]),train_accuracy=float(total[1]/total[2]),
        val_loss=float(val['loss']),val_accuracy=float(val['accuracy']),
        pared_epoca_s=time.perf_counter()-epoch_start,validacion_cpu_s=val_s))
    write_csv(run/'epocas.csv',epoch_rows)
    print('EPOCH',name,repetition,epoch_rows[-1],flush=True)
extra,_=net.export(optimizer=True);total_events(extra)
train_seconds=(time.perf_counter_ns()-train_start)/1e9
train_end_utc=utc()
assert net.steps==1440 and int(reference.optimizer.iterations.numpy())==1440
add('entrenamiento_'+name,train_seconds,inicio_utc=train_start_utc,fin_utc=train_end_utc)
add('validacion_anidada_'+name,validations)
for phase,seconds in events.items():add(phase,seconds,'evento_device')
print('ENTRENAMIENTO CUDA REAL',name,'corrida',repetition,'segundos',train_seconds,
    '| incluye staging, máscaras, H2D, GPU, D2H y validación CPU; pasos',net.steps)


In [ ]:
state('evaluacion_final_cpu',pasos=net.steps)
start=time.perf_counter_ns()
test=reference.evaluate(tx,ty,batch_size=32,verbose=0,return_dict=True)
eval_seconds=(time.perf_counter_ns()-start)/1e9
add('evaluacion_'+name,eval_seconds)
metrics=dict(modelo=name,repeticion=repetition,test={k:float(v) for k,v in test.items()},epocas=15,pasos=1440,
    entrenamiento_s=train_seconds,evaluacion_cpu_s=eval_seconds,
    comparabilidad='Datos, estado inicial, arquitectura y pasos equivalentes; CPU histórica; RNG interno diferente')
write_json(run/'metricas.json',metrics);write_csv(run/'tiempos.csv',timings)
write_json(run/'eventos.json',dict(eventos_s=events,validacion_cpu_s=validations,export_cpu_pared_s=export_wall,
    nota='Intervalos stream anidados; no equivalen a tiempo activo exclusivo de kernels; no sumar al tiempo pared'))
write_json(run/'ordenes.json',dict(seed=seed,generador='PCG64',epocas_sha256=shuffle_hashes))
param_count=sum(p['value'].size for p in net.params)
bn_count=sum(op['C']*2 for op in net.ops if op['kind']=='bn')
write_json(run/'transferencias.json',dict(lotes=net.transfers[-1440:],modelo_d2h_copias=16,
    modelo_bytes_por_copia=(param_count+bn_count)*4,optimizador_final_bytes=param_count*8+8,
    control_d2h_bytes=1440*4,reinicios_metricas_h2d_bytes=15*12,
    nota='Copia final adicional de pesos incluida; sin restar costes'))
state('guardado_fuera_timing')
model_dir=ROOT/'data/EXP-012'/run.name;model_dir.mkdir(parents=True,exist_ok=False)
model_path=model_dir/(name+'-CUDA.keras');reference.save(model_path)
fingerprint=hashlib.sha256()
for variable in [*reference.variables,*reference.optimizer.variables]:
    array=variable.numpy();fingerprint.update(str(array.dtype).encode());fingerprint.update(str(array.shape).encode());fingerprint.update(array.tobytes())
write_json(run/'modelo-guardado.json',dict(ruta=model_path.relative_to(ROOT).as_posix(),sha256=sha(model_path),
    estado_canonico_sha256=fingerprint.hexdigest(),optimizer_iterations=1440))
write_json(run/'lanzamientos.json',dict(vector_block=256,bn_parcial_block=[32,8,1],
    bn_parcial_grid=[1,384,1],bn_final_block=256,bn_final_grid=[48,1,1],
    softmax_block=32,softmax_grid=[128,1,1],ultimo_batch=20,recursos=compiled_resources(),
    fc_finish_regs=str(finish_fc.get_regs_per_thread()),
    gemm='cuBLAS SGEMM FP32 PEDANTIC; jerarquía interna de biblioteca, no fijada por el equipo'))
state('completada',pasos=1440)
stop.set();monitor.join(timeout=15)
if repetition==1:
    try:capture.wait(timeout=8)
    except subprocess.TimeoutExpired:capture.terminate();capture.wait();print('Captura no disponible; revisar capturador.log')
    capture_log.close()
display(pd.DataFrame(timings).fillna(''))
display(pd.DataFrame(epoch_rows))
print('MÉTRICAS REALES',json.dumps(metrics,indent=2,ensure_ascii=False))
fig,axes=plt.subplots(1,2,figsize=(10,3.6),layout='constrained')
for ax,keys,title in zip(axes,[('train_loss','val_loss'),('train_accuracy','val_accuracy')],['Pérdida','Exactitud']):
    for key in keys:ax.plot(range(1,16),[r[key] for r in epoch_rows],label=key)
    ax.set(xlabel='Época',ylabel=title,title=name+' · corrida '+str(repetition));ax.legend();ax.grid(alpha=.2)
fig.savefig(run/'curvas.png',dpi=160);fig.savefig(run/'curvas.svg');mostrar_figura(fig)
resource=pd.read_csv(run/'recursos.csv')
fig,axes=plt.subplots(2,1,figsize=(10,5),sharex=True,layout='constrained')
axes[0].plot(resource.transcurrido_s,resource.cpu_procesos_pct,color='#0072B2',label='CPU proceso e hijos (100% por núcleo)')
axes[1].plot(resource.transcurrido_s,resource.gpu_util_pct,color='#D55E00',label='Actividad GPU (%)')
axes[0].set(ylabel='Uso CPU (%)');axes[1].set(ylabel='Actividad GPU (%)',xlabel='Tiempo desde inicio del RUN (s)',ylim=(0,105))
for ax in axes:ax.legend(loc='upper right');ax.grid(alpha=.2)
fig.suptitle(name+' · monitorización real, corrida '+str(repetition))
fig.savefig(run/'recursos.png',dpi=160);fig.savefig(run/'recursos.svg');mostrar_figura(fig)
for picture in sorted(run.glob('captura-*.png')):display(Image(filename=str(picture)))
all_results.append(dict(run_id=run.name,**metrics));write_json(ROOT/'resultados/paralelo/EXP-012/progreso.json',all_results)
net.stream.synchronize();net.blas.close()
del net,reference,inputs,x,y,vx,vy,tx,ty,wx,wy,wv,wvy,wt,wty,p,op
tf.keras.backend.clear_session();gc.collect();cuda.current_context().deallocations.clear()


### FC-A: corrida 5

RUN: `RUN-20261118T134514Z-fc-a-05`. Inicialización/restauración → preparación CPU → entrenamiento CUDA → evaluación CPU y evidencias.

In [ ]:
name='FC-A';repetition=5;run_id='RUN-20261118T134514Z-fc-a-05'
run=ROOT/'resultados/paralelo/EXP-012'/run_id
run.mkdir(exist_ok=False)
timings=[];epoch_rows=[];events={};shuffle_hashes=[]
state('inicializacion')
stop,monitor=iniciar_monitor(run)
if repetition==1:
    capture_log=(run/'capturador.log').open('x',encoding='utf8')
    capture=subprocess.Popen([sys.executable,'-X','utf8',str(ROOT/'scripts/capturar_fc_real.py'),
        '--run',str(run),'--tipo','cuda','--modelo',name,'--nombre',name.lower()],
        stdout=capture_log,stderr=subprocess.STDOUT)
init_start=time.perf_counter()
tf,reference,seed,initial=fc_reference(name)
assert sha(initial)==config['estados_iniciales'][name]['sha256']
net=FCNetwork(reference,seed)
reference.summary()
print('GPU REAL',net.info,'Arena bytes',net.bytes_allocated,'BN momentum',[op['momentum'] for op in net.ops if op['kind']=='bn'])
write_json(run/'hardware.json',dict(device=net.info,tensorflow=tf.__version__,
    tf_build=tf.sysconfig.get_build_info(),cpu_tensor_device=tf.constant([1.]).device,
    tf_visible=[str(d) for d in tf.config.get_visible_devices()],cublas_version=net.blas.version,
    cublas_dll=str(net.blas.path),cublas_sha256=sha(net.blas.path),buffers_bytes=net.bytes_allocated,
    parametros=sum(p['value'].size for p in net.params),initial_checkpoint_sha256=sha(initial)))
assert not tf.config.get_visible_devices('GPU')
wx,wy=load_images(manifest['train'][:148],data_root,1)
wv,wvy=load_images(manifest['val'][:128],data_root,1)
wt,wty=load_images(manifest['test'][:32],data_root,1)
net.train_step(wx[:128],wy[:128],0,0);net.train_step(wx[128:],wy[128:],0,1)
reference.evaluate(wv,wvy,batch_size=128,verbose=0)
reference.evaluate(wt,wty,batch_size=32,verbose=0)
net.restore()
for p in net.params:
    assert np.array_equal(p['value'].copy_to_host(),p['initial'].ravel())
    assert not np.any(p['m'].copy_to_host()) and not np.any(p['v'].copy_to_host())
for op in net.ops:
    if op['kind']=='bn':
        assert np.array_equal(op['moving_mean'].copy_to_host(),op['initial_mean'])
        assert np.array_equal(op['moving_var'].copy_to_host(),op['initial_var'])
assert net.counter.copy_to_host()[0]==0
write_json(run/'warmup.json',dict(duracion_s=time.perf_counter()-init_start,train=148,val=128,test=32,
    restauracion_verificada=True,excluido_timing=True))
print('Warmup nuevo FC terminado y estado inicial restaurado; no incluido en entrenamiento.')


In [ ]:
state('preparacion_cpu',workers=6)
start=time.perf_counter()
inputs={split:load_images(manifest[split],data_root,6) for split in ['train','val','test']}
add('preparacion_cpu',time.perf_counter()-start)
tensors={}
for split,(images,labels) in inputs.items():
    a=hashlib.sha256(memoryview(images).cast('B')).hexdigest()
    b=hashlib.sha256(memoryview(labels).cast('B')).hexdigest()
    assert a==expected[split]['images_sha256'] and b==expected[split]['labels_sha256']
    tensors[split]=dict(images_sha256=a,labels_sha256=b,shape=list(images.shape),dtype=str(images.dtype))
write_json(run/'tensores.json',tensors)
x,y=inputs['train'];vx,vy=inputs['val'];tx,ty=inputs['test']
print('Preparación SOLO CPU (6 workers); bytes idénticos a EXP-002:',tensors)


In [ ]:
state('entrenamiento_'+name,epoca=1,pasos=0)
train_start_utc=utc();train_start=time.perf_counter_ns()
validations=0.;export_wall=0.
with net.timed('h2d_inicial'):net.restore()
total_events(net.drain_events())
rng=np.random.Generator(np.random.PCG64(seed))
for epoch in range(15):
    epoch_start=time.perf_counter()
    state('entrenamiento_'+name,epoca=epoch+1,pasos=net.steps)
    net.reset_metrics()
    order=rng.permutation(len(x))
    shuffle_hashes.append(hashlib.sha256(order.astype(np.int64).tobytes()).hexdigest())
    for j,begin in enumerate(range(0,len(x),128)):
        indices=order[begin:begin+128]
        total_events(net.train_step(x[indices],y[indices],epoch,j))
    export_start=time.perf_counter()
    extra,total=net.export();total_events(extra)
    export_wall+=time.perf_counter()-export_start
    assert int(total[2])==12180
    state('validacion_cpu',epoca=epoch+1,pasos=net.steps)
    val_start=time.perf_counter()
    val=reference.evaluate(vx,vy,batch_size=128,verbose=0,return_dict=True)
    val_s=time.perf_counter()-val_start;validations+=val_s
    epoch_rows.append(dict(modelo=name,repeticion=repetition,epoca=epoch+1,pasos_acumulados=net.steps,
        train_loss=float(total[0]/total[2]),train_accuracy=float(total[1]/total[2]),
        val_loss=float(val['loss']),val_accuracy=float(val['accuracy']),
        pared_epoca_s=time.perf_counter()-epoch_start,validacion_cpu_s=val_s))
    write_csv(run/'epocas.csv',epoch_rows)
    print('EPOCH',name,repetition,epoch_rows[-1],flush=True)
extra,_=net.export(optimizer=True);total_events(extra)
train_seconds=(time.perf_counter_ns()-train_start)/1e9
train_end_utc=utc()
assert net.steps==1440 and int(reference.optimizer.iterations.numpy())==1440
add('entrenamiento_'+name,train_seconds,inicio_utc=train_start_utc,fin_utc=train_end_utc)
add('validacion_anidada_'+name,validations)
for phase,seconds in events.items():add(phase,seconds,'evento_device')
print('ENTRENAMIENTO CUDA REAL',name,'corrida',repetition,'segundos',train_seconds,
    '| incluye staging, máscaras, H2D, GPU, D2H y validación CPU; pasos',net.steps)


In [ ]:
state('evaluacion_final_cpu',pasos=net.steps)
start=time.perf_counter_ns()
test=reference.evaluate(tx,ty,batch_size=32,verbose=0,return_dict=True)
eval_seconds=(time.perf_counter_ns()-start)/1e9
add('evaluacion_'+name,eval_seconds)
metrics=dict(modelo=name,repeticion=repetition,test={k:float(v) for k,v in test.items()},epocas=15,pasos=1440,
    entrenamiento_s=train_seconds,evaluacion_cpu_s=eval_seconds,
    comparabilidad='Datos, estado inicial, arquitectura y pasos equivalentes; CPU histórica; RNG interno diferente')
write_json(run/'metricas.json',metrics);write_csv(run/'tiempos.csv',timings)
write_json(run/'eventos.json',dict(eventos_s=events,validacion_cpu_s=validations,export_cpu_pared_s=export_wall,
    nota='Intervalos stream anidados; no equivalen a tiempo activo exclusivo de kernels; no sumar al tiempo pared'))
write_json(run/'ordenes.json',dict(seed=seed,generador='PCG64',epocas_sha256=shuffle_hashes))
param_count=sum(p['value'].size for p in net.params)
bn_count=sum(op['C']*2 for op in net.ops if op['kind']=='bn')
write_json(run/'transferencias.json',dict(lotes=net.transfers[-1440:],modelo_d2h_copias=16,
    modelo_bytes_por_copia=(param_count+bn_count)*4,optimizador_final_bytes=param_count*8+8,
    control_d2h_bytes=1440*4,reinicios_metricas_h2d_bytes=15*12,
    nota='Copia final adicional de pesos incluida; sin restar costes'))
state('guardado_fuera_timing')
model_dir=ROOT/'data/EXP-012'/run.name;model_dir.mkdir(parents=True,exist_ok=False)
model_path=model_dir/(name+'-CUDA.keras');reference.save(model_path)
fingerprint=hashlib.sha256()
for variable in [*reference.variables,*reference.optimizer.variables]:
    array=variable.numpy();fingerprint.update(str(array.dtype).encode());fingerprint.update(str(array.shape).encode());fingerprint.update(array.tobytes())
write_json(run/'modelo-guardado.json',dict(ruta=model_path.relative_to(ROOT).as_posix(),sha256=sha(model_path),
    estado_canonico_sha256=fingerprint.hexdigest(),optimizer_iterations=1440))
write_json(run/'lanzamientos.json',dict(vector_block=256,bn_parcial_block=[32,8,1],
    bn_parcial_grid=[1,384,1],bn_final_block=256,bn_final_grid=[48,1,1],
    softmax_block=32,softmax_grid=[128,1,1],ultimo_batch=20,recursos=compiled_resources(),
    fc_finish_regs=str(finish_fc.get_regs_per_thread()),
    gemm='cuBLAS SGEMM FP32 PEDANTIC; jerarquía interna de biblioteca, no fijada por el equipo'))
state('completada',pasos=1440)
stop.set();monitor.join(timeout=15)
if repetition==1:
    try:capture.wait(timeout=8)
    except subprocess.TimeoutExpired:capture.terminate();capture.wait();print('Captura no disponible; revisar capturador.log')
    capture_log.close()
display(pd.DataFrame(timings).fillna(''))
display(pd.DataFrame(epoch_rows))
print('MÉTRICAS REALES',json.dumps(metrics,indent=2,ensure_ascii=False))
fig,axes=plt.subplots(1,2,figsize=(10,3.6),layout='constrained')
for ax,keys,title in zip(axes,[('train_loss','val_loss'),('train_accuracy','val_accuracy')],['Pérdida','Exactitud']):
    for key in keys:ax.plot(range(1,16),[r[key] for r in epoch_rows],label=key)
    ax.set(xlabel='Época',ylabel=title,title=name+' · corrida '+str(repetition));ax.legend();ax.grid(alpha=.2)
fig.savefig(run/'curvas.png',dpi=160);fig.savefig(run/'curvas.svg');mostrar_figura(fig)
resource=pd.read_csv(run/'recursos.csv')
fig,axes=plt.subplots(2,1,figsize=(10,5),sharex=True,layout='constrained')
axes[0].plot(resource.transcurrido_s,resource.cpu_procesos_pct,color='#0072B2',label='CPU proceso e hijos (100% por núcleo)')
axes[1].plot(resource.transcurrido_s,resource.gpu_util_pct,color='#D55E00',label='Actividad GPU (%)')
axes[0].set(ylabel='Uso CPU (%)');axes[1].set(ylabel='Actividad GPU (%)',xlabel='Tiempo desde inicio del RUN (s)',ylim=(0,105))
for ax in axes:ax.legend(loc='upper right');ax.grid(alpha=.2)
fig.suptitle(name+' · monitorización real, corrida '+str(repetition))
fig.savefig(run/'recursos.png',dpi=160);fig.savefig(run/'recursos.svg');mostrar_figura(fig)
for picture in sorted(run.glob('captura-*.png')):display(Image(filename=str(picture)))
all_results.append(dict(run_id=run.name,**metrics));write_json(ROOT/'resultados/paralelo/EXP-012/progreso.json',all_results)
net.stream.synchronize();net.blas.close()
del net,reference,inputs,x,y,vx,vy,tx,ty,wx,wy,wv,wvy,wt,wty,p,op
tf.keras.backend.clear_session();gc.collect();cuda.current_context().deallocations.clear()


## FC-B — cinco ejecuciones nuevas

Batch128, último lote20, 15épocas, 1440actualizaciones. La medida pared incluye transferencias y validación CPU; guardado y gráficas quedan fuera. La primera corrida incorpora captura en vivo; su coste no se resta.

### FC-B: corrida 1

RUN: `RUN-20261118T134514Z-fc-b-01`. Inicialización/restauración → preparación CPU → entrenamiento CUDA → evaluación CPU y evidencias.

In [ ]:
name='FC-B';repetition=1;run_id='RUN-20261118T134514Z-fc-b-01'
run=ROOT/'resultados/paralelo/EXP-012'/run_id
run.mkdir(exist_ok=False)
timings=[];epoch_rows=[];events={};shuffle_hashes=[]
state('inicializacion')
stop,monitor=iniciar_monitor(run)
if repetition==1:
    capture_log=(run/'capturador.log').open('x',encoding='utf8')
    capture=subprocess.Popen([sys.executable,'-X','utf8',str(ROOT/'scripts/capturar_fc_real.py'),
        '--run',str(run),'--tipo','cuda','--modelo',name,'--nombre',name.lower()],
        stdout=capture_log,stderr=subprocess.STDOUT)
init_start=time.perf_counter()
tf,reference,seed,initial=fc_reference(name)
assert sha(initial)==config['estados_iniciales'][name]['sha256']
net=FCNetwork(reference,seed)
reference.summary()
print('GPU REAL',net.info,'Arena bytes',net.bytes_allocated,'BN momentum',[op['momentum'] for op in net.ops if op['kind']=='bn'])
write_json(run/'hardware.json',dict(device=net.info,tensorflow=tf.__version__,
    tf_build=tf.sysconfig.get_build_info(),cpu_tensor_device=tf.constant([1.]).device,
    tf_visible=[str(d) for d in tf.config.get_visible_devices()],cublas_version=net.blas.version,
    cublas_dll=str(net.blas.path),cublas_sha256=sha(net.blas.path),buffers_bytes=net.bytes_allocated,
    parametros=sum(p['value'].size for p in net.params),initial_checkpoint_sha256=sha(initial)))
assert not tf.config.get_visible_devices('GPU')
wx,wy=load_images(manifest['train'][:148],data_root,1)
wv,wvy=load_images(manifest['val'][:128],data_root,1)
wt,wty=load_images(manifest['test'][:32],data_root,1)
net.train_step(wx[:128],wy[:128],0,0);net.train_step(wx[128:],wy[128:],0,1)
reference.evaluate(wv,wvy,batch_size=128,verbose=0)
reference.evaluate(wt,wty,batch_size=32,verbose=0)
net.restore()
for p in net.params:
    assert np.array_equal(p['value'].copy_to_host(),p['initial'].ravel())
    assert not np.any(p['m'].copy_to_host()) and not np.any(p['v'].copy_to_host())
for op in net.ops:
    if op['kind']=='bn':
        assert np.array_equal(op['moving_mean'].copy_to_host(),op['initial_mean'])
        assert np.array_equal(op['moving_var'].copy_to_host(),op['initial_var'])
assert net.counter.copy_to_host()[0]==0
write_json(run/'warmup.json',dict(duracion_s=time.perf_counter()-init_start,train=148,val=128,test=32,
    restauracion_verificada=True,excluido_timing=True))
print('Warmup nuevo FC terminado y estado inicial restaurado; no incluido en entrenamiento.')


In [ ]:
state('preparacion_cpu',workers=6)
start=time.perf_counter()
inputs={split:load_images(manifest[split],data_root,6) for split in ['train','val','test']}
add('preparacion_cpu',time.perf_counter()-start)
tensors={}
for split,(images,labels) in inputs.items():
    a=hashlib.sha256(memoryview(images).cast('B')).hexdigest()
    b=hashlib.sha256(memoryview(labels).cast('B')).hexdigest()
    assert a==expected[split]['images_sha256'] and b==expected[split]['labels_sha256']
    tensors[split]=dict(images_sha256=a,labels_sha256=b,shape=list(images.shape),dtype=str(images.dtype))
write_json(run/'tensores.json',tensors)
x,y=inputs['train'];vx,vy=inputs['val'];tx,ty=inputs['test']
print('Preparación SOLO CPU (6 workers); bytes idénticos a EXP-002:',tensors)


In [ ]:
state('entrenamiento_'+name,epoca=1,pasos=0)
train_start_utc=utc();train_start=time.perf_counter_ns()
validations=0.;export_wall=0.
with net.timed('h2d_inicial'):net.restore()
total_events(net.drain_events())
rng=np.random.Generator(np.random.PCG64(seed))
for epoch in range(15):
    epoch_start=time.perf_counter()
    state('entrenamiento_'+name,epoca=epoch+1,pasos=net.steps)
    net.reset_metrics()
    order=rng.permutation(len(x))
    shuffle_hashes.append(hashlib.sha256(order.astype(np.int64).tobytes()).hexdigest())
    for j,begin in enumerate(range(0,len(x),128)):
        indices=order[begin:begin+128]
        total_events(net.train_step(x[indices],y[indices],epoch,j))
    export_start=time.perf_counter()
    extra,total=net.export();total_events(extra)
    export_wall+=time.perf_counter()-export_start
    assert int(total[2])==12180
    state('validacion_cpu',epoca=epoch+1,pasos=net.steps)
    val_start=time.perf_counter()
    val=reference.evaluate(vx,vy,batch_size=128,verbose=0,return_dict=True)
    val_s=time.perf_counter()-val_start;validations+=val_s
    epoch_rows.append(dict(modelo=name,repeticion=repetition,epoca=epoch+1,pasos_acumulados=net.steps,
        train_loss=float(total[0]/total[2]),train_accuracy=float(total[1]/total[2]),
        val_loss=float(val['loss']),val_accuracy=float(val['accuracy']),
        pared_epoca_s=time.perf_counter()-epoch_start,validacion_cpu_s=val_s))
    write_csv(run/'epocas.csv',epoch_rows)
    print('EPOCH',name,repetition,epoch_rows[-1],flush=True)
extra,_=net.export(optimizer=True);total_events(extra)
train_seconds=(time.perf_counter_ns()-train_start)/1e9
train_end_utc=utc()
assert net.steps==1440 and int(reference.optimizer.iterations.numpy())==1440
add('entrenamiento_'+name,train_seconds,inicio_utc=train_start_utc,fin_utc=train_end_utc)
add('validacion_anidada_'+name,validations)
for phase,seconds in events.items():add(phase,seconds,'evento_device')
print('ENTRENAMIENTO CUDA REAL',name,'corrida',repetition,'segundos',train_seconds,
    '| incluye staging, máscaras, H2D, GPU, D2H y validación CPU; pasos',net.steps)


In [ ]:
state('evaluacion_final_cpu',pasos=net.steps)
start=time.perf_counter_ns()
test=reference.evaluate(tx,ty,batch_size=32,verbose=0,return_dict=True)
eval_seconds=(time.perf_counter_ns()-start)/1e9
add('evaluacion_'+name,eval_seconds)
metrics=dict(modelo=name,repeticion=repetition,test={k:float(v) for k,v in test.items()},epocas=15,pasos=1440,
    entrenamiento_s=train_seconds,evaluacion_cpu_s=eval_seconds,
    comparabilidad='Datos, estado inicial, arquitectura y pasos equivalentes; CPU histórica; RNG interno diferente')
write_json(run/'metricas.json',metrics);write_csv(run/'tiempos.csv',timings)
write_json(run/'eventos.json',dict(eventos_s=events,validacion_cpu_s=validations,export_cpu_pared_s=export_wall,
    nota='Intervalos stream anidados; no equivalen a tiempo activo exclusivo de kernels; no sumar al tiempo pared'))
write_json(run/'ordenes.json',dict(seed=seed,generador='PCG64',epocas_sha256=shuffle_hashes))
param_count=sum(p['value'].size for p in net.params)
bn_count=sum(op['C']*2 for op in net.ops if op['kind']=='bn')
write_json(run/'transferencias.json',dict(lotes=net.transfers[-1440:],modelo_d2h_copias=16,
    modelo_bytes_por_copia=(param_count+bn_count)*4,optimizador_final_bytes=param_count*8+8,
    control_d2h_bytes=1440*4,reinicios_metricas_h2d_bytes=15*12,
    nota='Copia final adicional de pesos incluida; sin restar costes'))
state('guardado_fuera_timing')
model_dir=ROOT/'data/EXP-012'/run.name;model_dir.mkdir(parents=True,exist_ok=False)
model_path=model_dir/(name+'-CUDA.keras');reference.save(model_path)
fingerprint=hashlib.sha256()
for variable in [*reference.variables,*reference.optimizer.variables]:
    array=variable.numpy();fingerprint.update(str(array.dtype).encode());fingerprint.update(str(array.shape).encode());fingerprint.update(array.tobytes())
write_json(run/'modelo-guardado.json',dict(ruta=model_path.relative_to(ROOT).as_posix(),sha256=sha(model_path),
    estado_canonico_sha256=fingerprint.hexdigest(),optimizer_iterations=1440))
write_json(run/'lanzamientos.json',dict(vector_block=256,bn_parcial_block=[32,8,1],
    bn_parcial_grid=[1,384,1],bn_final_block=256,bn_final_grid=[48,1,1],
    softmax_block=32,softmax_grid=[128,1,1],ultimo_batch=20,recursos=compiled_resources(),
    fc_finish_regs=str(finish_fc.get_regs_per_thread()),
    gemm='cuBLAS SGEMM FP32 PEDANTIC; jerarquía interna de biblioteca, no fijada por el equipo'))
state('completada',pasos=1440)
stop.set();monitor.join(timeout=15)
if repetition==1:
    try:capture.wait(timeout=8)
    except subprocess.TimeoutExpired:capture.terminate();capture.wait();print('Captura no disponible; revisar capturador.log')
    capture_log.close()
display(pd.DataFrame(timings).fillna(''))
display(pd.DataFrame(epoch_rows))
print('MÉTRICAS REALES',json.dumps(metrics,indent=2,ensure_ascii=False))
fig,axes=plt.subplots(1,2,figsize=(10,3.6),layout='constrained')
for ax,keys,title in zip(axes,[('train_loss','val_loss'),('train_accuracy','val_accuracy')],['Pérdida','Exactitud']):
    for key in keys:ax.plot(range(1,16),[r[key] for r in epoch_rows],label=key)
    ax.set(xlabel='Época',ylabel=title,title=name+' · corrida '+str(repetition));ax.legend();ax.grid(alpha=.2)
fig.savefig(run/'curvas.png',dpi=160);fig.savefig(run/'curvas.svg');mostrar_figura(fig)
resource=pd.read_csv(run/'recursos.csv')
fig,axes=plt.subplots(2,1,figsize=(10,5),sharex=True,layout='constrained')
axes[0].plot(resource.transcurrido_s,resource.cpu_procesos_pct,color='#0072B2',label='CPU proceso e hijos (100% por núcleo)')
axes[1].plot(resource.transcurrido_s,resource.gpu_util_pct,color='#D55E00',label='Actividad GPU (%)')
axes[0].set(ylabel='Uso CPU (%)');axes[1].set(ylabel='Actividad GPU (%)',xlabel='Tiempo desde inicio del RUN (s)',ylim=(0,105))
for ax in axes:ax.legend(loc='upper right');ax.grid(alpha=.2)
fig.suptitle(name+' · monitorización real, corrida '+str(repetition))
fig.savefig(run/'recursos.png',dpi=160);fig.savefig(run/'recursos.svg');mostrar_figura(fig)
for picture in sorted(run.glob('captura-*.png')):display(Image(filename=str(picture)))
all_results.append(dict(run_id=run.name,**metrics));write_json(ROOT/'resultados/paralelo/EXP-012/progreso.json',all_results)
net.stream.synchronize();net.blas.close()
del net,reference,inputs,x,y,vx,vy,tx,ty,wx,wy,wv,wvy,wt,wty,p,op
tf.keras.backend.clear_session();gc.collect();cuda.current_context().deallocations.clear()


### FC-B: corrida 2

RUN: `RUN-20261118T134514Z-fc-b-02`. Inicialización/restauración → preparación CPU → entrenamiento CUDA → evaluación CPU y evidencias.

In [ ]:
name='FC-B';repetition=2;run_id='RUN-20261118T134514Z-fc-b-02'
run=ROOT/'resultados/paralelo/EXP-012'/run_id
run.mkdir(exist_ok=False)
timings=[];epoch_rows=[];events={};shuffle_hashes=[]
state('inicializacion')
stop,monitor=iniciar_monitor(run)
if repetition==1:
    capture_log=(run/'capturador.log').open('x',encoding='utf8')
    capture=subprocess.Popen([sys.executable,'-X','utf8',str(ROOT/'scripts/capturar_fc_real.py'),
        '--run',str(run),'--tipo','cuda','--modelo',name,'--nombre',name.lower()],
        stdout=capture_log,stderr=subprocess.STDOUT)
init_start=time.perf_counter()
tf,reference,seed,initial=fc_reference(name)
assert sha(initial)==config['estados_iniciales'][name]['sha256']
net=FCNetwork(reference,seed)
reference.summary()
print('GPU REAL',net.info,'Arena bytes',net.bytes_allocated,'BN momentum',[op['momentum'] for op in net.ops if op['kind']=='bn'])
write_json(run/'hardware.json',dict(device=net.info,tensorflow=tf.__version__,
    tf_build=tf.sysconfig.get_build_info(),cpu_tensor_device=tf.constant([1.]).device,
    tf_visible=[str(d) for d in tf.config.get_visible_devices()],cublas_version=net.blas.version,
    cublas_dll=str(net.blas.path),cublas_sha256=sha(net.blas.path),buffers_bytes=net.bytes_allocated,
    parametros=sum(p['value'].size for p in net.params),initial_checkpoint_sha256=sha(initial)))
assert not tf.config.get_visible_devices('GPU')
wx,wy=load_images(manifest['train'][:148],data_root,1)
wv,wvy=load_images(manifest['val'][:128],data_root,1)
wt,wty=load_images(manifest['test'][:32],data_root,1)
net.train_step(wx[:128],wy[:128],0,0);net.train_step(wx[128:],wy[128:],0,1)
reference.evaluate(wv,wvy,batch_size=128,verbose=0)
reference.evaluate(wt,wty,batch_size=32,verbose=0)
net.restore()
for p in net.params:
    assert np.array_equal(p['value'].copy_to_host(),p['initial'].ravel())
    assert not np.any(p['m'].copy_to_host()) and not np.any(p['v'].copy_to_host())
for op in net.ops:
    if op['kind']=='bn':
        assert np.array_equal(op['moving_mean'].copy_to_host(),op['initial_mean'])
        assert np.array_equal(op['moving_var'].copy_to_host(),op['initial_var'])
assert net.counter.copy_to_host()[0]==0
write_json(run/'warmup.json',dict(duracion_s=time.perf_counter()-init_start,train=148,val=128,test=32,
    restauracion_verificada=True,excluido_timing=True))
print('Warmup nuevo FC terminado y estado inicial restaurado; no incluido en entrenamiento.')


In [ ]:
state('preparacion_cpu',workers=6)
start=time.perf_counter()
inputs={split:load_images(manifest[split],data_root,6) for split in ['train','val','test']}
add('preparacion_cpu',time.perf_counter()-start)
tensors={}
for split,(images,labels) in inputs.items():
    a=hashlib.sha256(memoryview(images).cast('B')).hexdigest()
    b=hashlib.sha256(memoryview(labels).cast('B')).hexdigest()
    assert a==expected[split]['images_sha256'] and b==expected[split]['labels_sha256']
    tensors[split]=dict(images_sha256=a,labels_sha256=b,shape=list(images.shape),dtype=str(images.dtype))
write_json(run/'tensores.json',tensors)
x,y=inputs['train'];vx,vy=inputs['val'];tx,ty=inputs['test']
print('Preparación SOLO CPU (6 workers); bytes idénticos a EXP-002:',tensors)


In [ ]:
state('entrenamiento_'+name,epoca=1,pasos=0)
train_start_utc=utc();train_start=time.perf_counter_ns()
validations=0.;export_wall=0.
with net.timed('h2d_inicial'):net.restore()
total_events(net.drain_events())
rng=np.random.Generator(np.random.PCG64(seed))
for epoch in range(15):
    epoch_start=time.perf_counter()
    state('entrenamiento_'+name,epoca=epoch+1,pasos=net.steps)
    net.reset_metrics()
    order=rng.permutation(len(x))
    shuffle_hashes.append(hashlib.sha256(order.astype(np.int64).tobytes()).hexdigest())
    for j,begin in enumerate(range(0,len(x),128)):
        indices=order[begin:begin+128]
        total_events(net.train_step(x[indices],y[indices],epoch,j))
    export_start=time.perf_counter()
    extra,total=net.export();total_events(extra)
    export_wall+=time.perf_counter()-export_start
    assert int(total[2])==12180
    state('validacion_cpu',epoca=epoch+1,pasos=net.steps)
    val_start=time.perf_counter()
    val=reference.evaluate(vx,vy,batch_size=128,verbose=0,return_dict=True)
    val_s=time.perf_counter()-val_start;validations+=val_s
    epoch_rows.append(dict(modelo=name,repeticion=repetition,epoca=epoch+1,pasos_acumulados=net.steps,
        train_loss=float(total[0]/total[2]),train_accuracy=float(total[1]/total[2]),
        val_loss=float(val['loss']),val_accuracy=float(val['accuracy']),
        pared_epoca_s=time.perf_counter()-epoch_start,validacion_cpu_s=val_s))
    write_csv(run/'epocas.csv',epoch_rows)
    print('EPOCH',name,repetition,epoch_rows[-1],flush=True)
extra,_=net.export(optimizer=True);total_events(extra)
train_seconds=(time.perf_counter_ns()-train_start)/1e9
train_end_utc=utc()
assert net.steps==1440 and int(reference.optimizer.iterations.numpy())==1440
add('entrenamiento_'+name,train_seconds,inicio_utc=train_start_utc,fin_utc=train_end_utc)
add('validacion_anidada_'+name,validations)
for phase,seconds in events.items():add(phase,seconds,'evento_device')
print('ENTRENAMIENTO CUDA REAL',name,'corrida',repetition,'segundos',train_seconds,
    '| incluye staging, máscaras, H2D, GPU, D2H y validación CPU; pasos',net.steps)


In [ ]:
state('evaluacion_final_cpu',pasos=net.steps)
start=time.perf_counter_ns()
test=reference.evaluate(tx,ty,batch_size=32,verbose=0,return_dict=True)
eval_seconds=(time.perf_counter_ns()-start)/1e9
add('evaluacion_'+name,eval_seconds)
metrics=dict(modelo=name,repeticion=repetition,test={k:float(v) for k,v in test.items()},epocas=15,pasos=1440,
    entrenamiento_s=train_seconds,evaluacion_cpu_s=eval_seconds,
    comparabilidad='Datos, estado inicial, arquitectura y pasos equivalentes; CPU histórica; RNG interno diferente')
write_json(run/'metricas.json',metrics);write_csv(run/'tiempos.csv',timings)
write_json(run/'eventos.json',dict(eventos_s=events,validacion_cpu_s=validations,export_cpu_pared_s=export_wall,
    nota='Intervalos stream anidados; no equivalen a tiempo activo exclusivo de kernels; no sumar al tiempo pared'))
write_json(run/'ordenes.json',dict(seed=seed,generador='PCG64',epocas_sha256=shuffle_hashes))
param_count=sum(p['value'].size for p in net.params)
bn_count=sum(op['C']*2 for op in net.ops if op['kind']=='bn')
write_json(run/'transferencias.json',dict(lotes=net.transfers[-1440:],modelo_d2h_copias=16,
    modelo_bytes_por_copia=(param_count+bn_count)*4,optimizador_final_bytes=param_count*8+8,
    control_d2h_bytes=1440*4,reinicios_metricas_h2d_bytes=15*12,
    nota='Copia final adicional de pesos incluida; sin restar costes'))
state('guardado_fuera_timing')
model_dir=ROOT/'data/EXP-012'/run.name;model_dir.mkdir(parents=True,exist_ok=False)
model_path=model_dir/(name+'-CUDA.keras');reference.save(model_path)
fingerprint=hashlib.sha256()
for variable in [*reference.variables,*reference.optimizer.variables]:
    array=variable.numpy();fingerprint.update(str(array.dtype).encode());fingerprint.update(str(array.shape).encode());fingerprint.update(array.tobytes())
write_json(run/'modelo-guardado.json',dict(ruta=model_path.relative_to(ROOT).as_posix(),sha256=sha(model_path),
    estado_canonico_sha256=fingerprint.hexdigest(),optimizer_iterations=1440))
write_json(run/'lanzamientos.json',dict(vector_block=256,bn_parcial_block=[32,8,1],
    bn_parcial_grid=[1,384,1],bn_final_block=256,bn_final_grid=[48,1,1],
    softmax_block=32,softmax_grid=[128,1,1],ultimo_batch=20,recursos=compiled_resources(),
    fc_finish_regs=str(finish_fc.get_regs_per_thread()),
    gemm='cuBLAS SGEMM FP32 PEDANTIC; jerarquía interna de biblioteca, no fijada por el equipo'))
state('completada',pasos=1440)
stop.set();monitor.join(timeout=15)
if repetition==1:
    try:capture.wait(timeout=8)
    except subprocess.TimeoutExpired:capture.terminate();capture.wait();print('Captura no disponible; revisar capturador.log')
    capture_log.close()
display(pd.DataFrame(timings).fillna(''))
display(pd.DataFrame(epoch_rows))
print('MÉTRICAS REALES',json.dumps(metrics,indent=2,ensure_ascii=False))
fig,axes=plt.subplots(1,2,figsize=(10,3.6),layout='constrained')
for ax,keys,title in zip(axes,[('train_loss','val_loss'),('train_accuracy','val_accuracy')],['Pérdida','Exactitud']):
    for key in keys:ax.plot(range(1,16),[r[key] for r in epoch_rows],label=key)
    ax.set(xlabel='Época',ylabel=title,title=name+' · corrida '+str(repetition));ax.legend();ax.grid(alpha=.2)
fig.savefig(run/'curvas.png',dpi=160);fig.savefig(run/'curvas.svg');mostrar_figura(fig)
resource=pd.read_csv(run/'recursos.csv')
fig,axes=plt.subplots(2,1,figsize=(10,5),sharex=True,layout='constrained')
axes[0].plot(resource.transcurrido_s,resource.cpu_procesos_pct,color='#0072B2',label='CPU proceso e hijos (100% por núcleo)')
axes[1].plot(resource.transcurrido_s,resource.gpu_util_pct,color='#D55E00',label='Actividad GPU (%)')
axes[0].set(ylabel='Uso CPU (%)');axes[1].set(ylabel='Actividad GPU (%)',xlabel='Tiempo desde inicio del RUN (s)',ylim=(0,105))
for ax in axes:ax.legend(loc='upper right');ax.grid(alpha=.2)
fig.suptitle(name+' · monitorización real, corrida '+str(repetition))
fig.savefig(run/'recursos.png',dpi=160);fig.savefig(run/'recursos.svg');mostrar_figura(fig)
for picture in sorted(run.glob('captura-*.png')):display(Image(filename=str(picture)))
all_results.append(dict(run_id=run.name,**metrics));write_json(ROOT/'resultados/paralelo/EXP-012/progreso.json',all_results)
net.stream.synchronize();net.blas.close()
del net,reference,inputs,x,y,vx,vy,tx,ty,wx,wy,wv,wvy,wt,wty,p,op
tf.keras.backend.clear_session();gc.collect();cuda.current_context().deallocations.clear()


### FC-B: corrida 3

RUN: `RUN-20261118T134514Z-fc-b-03`. Inicialización/restauración → preparación CPU → entrenamiento CUDA → evaluación CPU y evidencias.

In [ ]:
name='FC-B';repetition=3;run_id='RUN-20261118T134514Z-fc-b-03'
run=ROOT/'resultados/paralelo/EXP-012'/run_id
run.mkdir(exist_ok=False)
timings=[];epoch_rows=[];events={};shuffle_hashes=[]
state('inicializacion')
stop,monitor=iniciar_monitor(run)
if repetition==1:
    capture_log=(run/'capturador.log').open('x',encoding='utf8')
    capture=subprocess.Popen([sys.executable,'-X','utf8',str(ROOT/'scripts/capturar_fc_real.py'),
        '--run',str(run),'--tipo','cuda','--modelo',name,'--nombre',name.lower()],
        stdout=capture_log,stderr=subprocess.STDOUT)
init_start=time.perf_counter()
tf,reference,seed,initial=fc_reference(name)
assert sha(initial)==config['estados_iniciales'][name]['sha256']
net=FCNetwork(reference,seed)
reference.summary()
print('GPU REAL',net.info,'Arena bytes',net.bytes_allocated,'BN momentum',[op['momentum'] for op in net.ops if op['kind']=='bn'])
write_json(run/'hardware.json',dict(device=net.info,tensorflow=tf.__version__,
    tf_build=tf.sysconfig.get_build_info(),cpu_tensor_device=tf.constant([1.]).device,
    tf_visible=[str(d) for d in tf.config.get_visible_devices()],cublas_version=net.blas.version,
    cublas_dll=str(net.blas.path),cublas_sha256=sha(net.blas.path),buffers_bytes=net.bytes_allocated,
    parametros=sum(p['value'].size for p in net.params),initial_checkpoint_sha256=sha(initial)))
assert not tf.config.get_visible_devices('GPU')
wx,wy=load_images(manifest['train'][:148],data_root,1)
wv,wvy=load_images(manifest['val'][:128],data_root,1)
wt,wty=load_images(manifest['test'][:32],data_root,1)
net.train_step(wx[:128],wy[:128],0,0);net.train_step(wx[128:],wy[128:],0,1)
reference.evaluate(wv,wvy,batch_size=128,verbose=0)
reference.evaluate(wt,wty,batch_size=32,verbose=0)
net.restore()
for p in net.params:
    assert np.array_equal(p['value'].copy_to_host(),p['initial'].ravel())
    assert not np.any(p['m'].copy_to_host()) and not np.any(p['v'].copy_to_host())
for op in net.ops:
    if op['kind']=='bn':
        assert np.array_equal(op['moving_mean'].copy_to_host(),op['initial_mean'])
        assert np.array_equal(op['moving_var'].copy_to_host(),op['initial_var'])
assert net.counter.copy_to_host()[0]==0
write_json(run/'warmup.json',dict(duracion_s=time.perf_counter()-init_start,train=148,val=128,test=32,
    restauracion_verificada=True,excluido_timing=True))
print('Warmup nuevo FC terminado y estado inicial restaurado; no incluido en entrenamiento.')


In [ ]:
state('preparacion_cpu',workers=6)
start=time.perf_counter()
inputs={split:load_images(manifest[split],data_root,6) for split in ['train','val','test']}
add('preparacion_cpu',time.perf_counter()-start)
tensors={}
for split,(images,labels) in inputs.items():
    a=hashlib.sha256(memoryview(images).cast('B')).hexdigest()
    b=hashlib.sha256(memoryview(labels).cast('B')).hexdigest()
    assert a==expected[split]['images_sha256'] and b==expected[split]['labels_sha256']
    tensors[split]=dict(images_sha256=a,labels_sha256=b,shape=list(images.shape),dtype=str(images.dtype))
write_json(run/'tensores.json',tensors)
x,y=inputs['train'];vx,vy=inputs['val'];tx,ty=inputs['test']
print('Preparación SOLO CPU (6 workers); bytes idénticos a EXP-002:',tensors)


In [ ]:
state('entrenamiento_'+name,epoca=1,pasos=0)
train_start_utc=utc();train_start=time.perf_counter_ns()
validations=0.;export_wall=0.
with net.timed('h2d_inicial'):net.restore()
total_events(net.drain_events())
rng=np.random.Generator(np.random.PCG64(seed))
for epoch in range(15):
    epoch_start=time.perf_counter()
    state('entrenamiento_'+name,epoca=epoch+1,pasos=net.steps)
    net.reset_metrics()
    order=rng.permutation(len(x))
    shuffle_hashes.append(hashlib.sha256(order.astype(np.int64).tobytes()).hexdigest())
    for j,begin in enumerate(range(0,len(x),128)):
        indices=order[begin:begin+128]
        total_events(net.train_step(x[indices],y[indices],epoch,j))
    export_start=time.perf_counter()
    extra,total=net.export();total_events(extra)
    export_wall+=time.perf_counter()-export_start
    assert int(total[2])==12180
    state('validacion_cpu',epoca=epoch+1,pasos=net.steps)
    val_start=time.perf_counter()
    val=reference.evaluate(vx,vy,batch_size=128,verbose=0,return_dict=True)
    val_s=time.perf_counter()-val_start;validations+=val_s
    epoch_rows.append(dict(modelo=name,repeticion=repetition,epoca=epoch+1,pasos_acumulados=net.steps,
        train_loss=float(total[0]/total[2]),train_accuracy=float(total[1]/total[2]),
        val_loss=float(val['loss']),val_accuracy=float(val['accuracy']),
        pared_epoca_s=time.perf_counter()-epoch_start,validacion_cpu_s=val_s))
    write_csv(run/'epocas.csv',epoch_rows)
    print('EPOCH',name,repetition,epoch_rows[-1],flush=True)
extra,_=net.export(optimizer=True);total_events(extra)
train_seconds=(time.perf_counter_ns()-train_start)/1e9
train_end_utc=utc()
assert net.steps==1440 and int(reference.optimizer.iterations.numpy())==1440
add('entrenamiento_'+name,train_seconds,inicio_utc=train_start_utc,fin_utc=train_end_utc)
add('validacion_anidada_'+name,validations)
for phase,seconds in events.items():add(phase,seconds,'evento_device')
print('ENTRENAMIENTO CUDA REAL',name,'corrida',repetition,'segundos',train_seconds,
    '| incluye staging, máscaras, H2D, GPU, D2H y validación CPU; pasos',net.steps)


In [ ]:
state('evaluacion_final_cpu',pasos=net.steps)
start=time.perf_counter_ns()
test=reference.evaluate(tx,ty,batch_size=32,verbose=0,return_dict=True)
eval_seconds=(time.perf_counter_ns()-start)/1e9
add('evaluacion_'+name,eval_seconds)
metrics=dict(modelo=name,repeticion=repetition,test={k:float(v) for k,v in test.items()},epocas=15,pasos=1440,
    entrenamiento_s=train_seconds,evaluacion_cpu_s=eval_seconds,
    comparabilidad='Datos, estado inicial, arquitectura y pasos equivalentes; CPU histórica; RNG interno diferente')
write_json(run/'metricas.json',metrics);write_csv(run/'tiempos.csv',timings)
write_json(run/'eventos.json',dict(eventos_s=events,validacion_cpu_s=validations,export_cpu_pared_s=export_wall,
    nota='Intervalos stream anidados; no equivalen a tiempo activo exclusivo de kernels; no sumar al tiempo pared'))
write_json(run/'ordenes.json',dict(seed=seed,generador='PCG64',epocas_sha256=shuffle_hashes))
param_count=sum(p['value'].size for p in net.params)
bn_count=sum(op['C']*2 for op in net.ops if op['kind']=='bn')
write_json(run/'transferencias.json',dict(lotes=net.transfers[-1440:],modelo_d2h_copias=16,
    modelo_bytes_por_copia=(param_count+bn_count)*4,optimizador_final_bytes=param_count*8+8,
    control_d2h_bytes=1440*4,reinicios_metricas_h2d_bytes=15*12,
    nota='Copia final adicional de pesos incluida; sin restar costes'))
state('guardado_fuera_timing')
model_dir=ROOT/'data/EXP-012'/run.name;model_dir.mkdir(parents=True,exist_ok=False)
model_path=model_dir/(name+'-CUDA.keras');reference.save(model_path)
fingerprint=hashlib.sha256()
for variable in [*reference.variables,*reference.optimizer.variables]:
    array=variable.numpy();fingerprint.update(str(array.dtype).encode());fingerprint.update(str(array.shape).encode());fingerprint.update(array.tobytes())
write_json(run/'modelo-guardado.json',dict(ruta=model_path.relative_to(ROOT).as_posix(),sha256=sha(model_path),
    estado_canonico_sha256=fingerprint.hexdigest(),optimizer_iterations=1440))
write_json(run/'lanzamientos.json',dict(vector_block=256,bn_parcial_block=[32,8,1],
    bn_parcial_grid=[1,384,1],bn_final_block=256,bn_final_grid=[48,1,1],
    softmax_block=32,softmax_grid=[128,1,1],ultimo_batch=20,recursos=compiled_resources(),
    fc_finish_regs=str(finish_fc.get_regs_per_thread()),
    gemm='cuBLAS SGEMM FP32 PEDANTIC; jerarquía interna de biblioteca, no fijada por el equipo'))
state('completada',pasos=1440)
stop.set();monitor.join(timeout=15)
if repetition==1:
    try:capture.wait(timeout=8)
    except subprocess.TimeoutExpired:capture.terminate();capture.wait();print('Captura no disponible; revisar capturador.log')
    capture_log.close()
display(pd.DataFrame(timings).fillna(''))
display(pd.DataFrame(epoch_rows))
print('MÉTRICAS REALES',json.dumps(metrics,indent=2,ensure_ascii=False))
fig,axes=plt.subplots(1,2,figsize=(10,3.6),layout='constrained')
for ax,keys,title in zip(axes,[('train_loss','val_loss'),('train_accuracy','val_accuracy')],['Pérdida','Exactitud']):
    for key in keys:ax.plot(range(1,16),[r[key] for r in epoch_rows],label=key)
    ax.set(xlabel='Época',ylabel=title,title=name+' · corrida '+str(repetition));ax.legend();ax.grid(alpha=.2)
fig.savefig(run/'curvas.png',dpi=160);fig.savefig(run/'curvas.svg');mostrar_figura(fig)
resource=pd.read_csv(run/'recursos.csv')
fig,axes=plt.subplots(2,1,figsize=(10,5),sharex=True,layout='constrained')
axes[0].plot(resource.transcurrido_s,resource.cpu_procesos_pct,color='#0072B2',label='CPU proceso e hijos (100% por núcleo)')
axes[1].plot(resource.transcurrido_s,resource.gpu_util_pct,color='#D55E00',label='Actividad GPU (%)')
axes[0].set(ylabel='Uso CPU (%)');axes[1].set(ylabel='Actividad GPU (%)',xlabel='Tiempo desde inicio del RUN (s)',ylim=(0,105))
for ax in axes:ax.legend(loc='upper right');ax.grid(alpha=.2)
fig.suptitle(name+' · monitorización real, corrida '+str(repetition))
fig.savefig(run/'recursos.png',dpi=160);fig.savefig(run/'recursos.svg');mostrar_figura(fig)
for picture in sorted(run.glob('captura-*.png')):display(Image(filename=str(picture)))
all_results.append(dict(run_id=run.name,**metrics));write_json(ROOT/'resultados/paralelo/EXP-012/progreso.json',all_results)
net.stream.synchronize();net.blas.close()
del net,reference,inputs,x,y,vx,vy,tx,ty,wx,wy,wv,wvy,wt,wty,p,op
tf.keras.backend.clear_session();gc.collect();cuda.current_context().deallocations.clear()


### FC-B: corrida 4

RUN: `RUN-20261118T134514Z-fc-b-04`. Inicialización/restauración → preparación CPU → entrenamiento CUDA → evaluación CPU y evidencias.

In [ ]:
name='FC-B';repetition=4;run_id='RUN-20261118T134514Z-fc-b-04'
run=ROOT/'resultados/paralelo/EXP-012'/run_id
run.mkdir(exist_ok=False)
timings=[];epoch_rows=[];events={};shuffle_hashes=[]
state('inicializacion')
stop,monitor=iniciar_monitor(run)
if repetition==1:
    capture_log=(run/'capturador.log').open('x',encoding='utf8')
    capture=subprocess.Popen([sys.executable,'-X','utf8',str(ROOT/'scripts/capturar_fc_real.py'),
        '--run',str(run),'--tipo','cuda','--modelo',name,'--nombre',name.lower()],
        stdout=capture_log,stderr=subprocess.STDOUT)
init_start=time.perf_counter()
tf,reference,seed,initial=fc_reference(name)
assert sha(initial)==config['estados_iniciales'][name]['sha256']
net=FCNetwork(reference,seed)
reference.summary()
print('GPU REAL',net.info,'Arena bytes',net.bytes_allocated,'BN momentum',[op['momentum'] for op in net.ops if op['kind']=='bn'])
write_json(run/'hardware.json',dict(device=net.info,tensorflow=tf.__version__,
    tf_build=tf.sysconfig.get_build_info(),cpu_tensor_device=tf.constant([1.]).device,
    tf_visible=[str(d) for d in tf.config.get_visible_devices()],cublas_version=net.blas.version,
    cublas_dll=str(net.blas.path),cublas_sha256=sha(net.blas.path),buffers_bytes=net.bytes_allocated,
    parametros=sum(p['value'].size for p in net.params),initial_checkpoint_sha256=sha(initial)))
assert not tf.config.get_visible_devices('GPU')
wx,wy=load_images(manifest['train'][:148],data_root,1)
wv,wvy=load_images(manifest['val'][:128],data_root,1)
wt,wty=load_images(manifest['test'][:32],data_root,1)
net.train_step(wx[:128],wy[:128],0,0);net.train_step(wx[128:],wy[128:],0,1)
reference.evaluate(wv,wvy,batch_size=128,verbose=0)
reference.evaluate(wt,wty,batch_size=32,verbose=0)
net.restore()
for p in net.params:
    assert np.array_equal(p['value'].copy_to_host(),p['initial'].ravel())
    assert not np.any(p['m'].copy_to_host()) and not np.any(p['v'].copy_to_host())
for op in net.ops:
    if op['kind']=='bn':
        assert np.array_equal(op['moving_mean'].copy_to_host(),op['initial_mean'])
        assert np.array_equal(op['moving_var'].copy_to_host(),op['initial_var'])
assert net.counter.copy_to_host()[0]==0
write_json(run/'warmup.json',dict(duracion_s=time.perf_counter()-init_start,train=148,val=128,test=32,
    restauracion_verificada=True,excluido_timing=True))
print('Warmup nuevo FC terminado y estado inicial restaurado; no incluido en entrenamiento.')


In [ ]:
state('preparacion_cpu',workers=6)
start=time.perf_counter()
inputs={split:load_images(manifest[split],data_root,6) for split in ['train','val','test']}
add('preparacion_cpu',time.perf_counter()-start)
tensors={}
for split,(images,labels) in inputs.items():
    a=hashlib.sha256(memoryview(images).cast('B')).hexdigest()
    b=hashlib.sha256(memoryview(labels).cast('B')).hexdigest()
    assert a==expected[split]['images_sha256'] and b==expected[split]['labels_sha256']
    tensors[split]=dict(images_sha256=a,labels_sha256=b,shape=list(images.shape),dtype=str(images.dtype))
write_json(run/'tensores.json',tensors)
x,y=inputs['train'];vx,vy=inputs['val'];tx,ty=inputs['test']
print('Preparación SOLO CPU (6 workers); bytes idénticos a EXP-002:',tensors)


In [ ]:
state('entrenamiento_'+name,epoca=1,pasos=0)
train_start_utc=utc();train_start=time.perf_counter_ns()
validations=0.;export_wall=0.
with net.timed('h2d_inicial'):net.restore()
total_events(net.drain_events())
rng=np.random.Generator(np.random.PCG64(seed))
for epoch in range(15):
    epoch_start=time.perf_counter()
    state('entrenamiento_'+name,epoca=epoch+1,pasos=net.steps)
    net.reset_metrics()
    order=rng.permutation(len(x))
    shuffle_hashes.append(hashlib.sha256(order.astype(np.int64).tobytes()).hexdigest())
    for j,begin in enumerate(range(0,len(x),128)):
        indices=order[begin:begin+128]
        total_events(net.train_step(x[indices],y[indices],epoch,j))
    export_start=time.perf_counter()
    extra,total=net.export();total_events(extra)
    export_wall+=time.perf_counter()-export_start
    assert int(total[2])==12180
    state('validacion_cpu',epoca=epoch+1,pasos=net.steps)
    val_start=time.perf_counter()
    val=reference.evaluate(vx,vy,batch_size=128,verbose=0,return_dict=True)
    val_s=time.perf_counter()-val_start;validations+=val_s
    epoch_rows.append(dict(modelo=name,repeticion=repetition,epoca=epoch+1,pasos_acumulados=net.steps,
        train_loss=float(total[0]/total[2]),train_accuracy=float(total[1]/total[2]),
        val_loss=float(val['loss']),val_accuracy=float(val['accuracy']),
        pared_epoca_s=time.perf_counter()-epoch_start,validacion_cpu_s=val_s))
    write_csv(run/'epocas.csv',epoch_rows)
    print('EPOCH',name,repetition,epoch_rows[-1],flush=True)
extra,_=net.export(optimizer=True);total_events(extra)
train_seconds=(time.perf_counter_ns()-train_start)/1e9
train_end_utc=utc()
assert net.steps==1440 and int(reference.optimizer.iterations.numpy())==1440
add('entrenamiento_'+name,train_seconds,inicio_utc=train_start_utc,fin_utc=train_end_utc)
add('validacion_anidada_'+name,validations)
for phase,seconds in events.items():add(phase,seconds,'evento_device')
print('ENTRENAMIENTO CUDA REAL',name,'corrida',repetition,'segundos',train_seconds,
    '| incluye staging, máscaras, H2D, GPU, D2H y validación CPU; pasos',net.steps)


In [ ]:
state('evaluacion_final_cpu',pasos=net.steps)
start=time.perf_counter_ns()
test=reference.evaluate(tx,ty,batch_size=32,verbose=0,return_dict=True)
eval_seconds=(time.perf_counter_ns()-start)/1e9
add('evaluacion_'+name,eval_seconds)
metrics=dict(modelo=name,repeticion=repetition,test={k:float(v) for k,v in test.items()},epocas=15,pasos=1440,
    entrenamiento_s=train_seconds,evaluacion_cpu_s=eval_seconds,
    comparabilidad='Datos, estado inicial, arquitectura y pasos equivalentes; CPU histórica; RNG interno diferente')
write_json(run/'metricas.json',metrics);write_csv(run/'tiempos.csv',timings)
write_json(run/'eventos.json',dict(eventos_s=events,validacion_cpu_s=validations,export_cpu_pared_s=export_wall,
    nota='Intervalos stream anidados; no equivalen a tiempo activo exclusivo de kernels; no sumar al tiempo pared'))
write_json(run/'ordenes.json',dict(seed=seed,generador='PCG64',epocas_sha256=shuffle_hashes))
param_count=sum(p['value'].size for p in net.params)
bn_count=sum(op['C']*2 for op in net.ops if op['kind']=='bn')
write_json(run/'transferencias.json',dict(lotes=net.transfers[-1440:],modelo_d2h_copias=16,
    modelo_bytes_por_copia=(param_count+bn_count)*4,optimizador_final_bytes=param_count*8+8,
    control_d2h_bytes=1440*4,reinicios_metricas_h2d_bytes=15*12,
    nota='Copia final adicional de pesos incluida; sin restar costes'))
state('guardado_fuera_timing')
model_dir=ROOT/'data/EXP-012'/run.name;model_dir.mkdir(parents=True,exist_ok=False)
model_path=model_dir/(name+'-CUDA.keras');reference.save(model_path)
fingerprint=hashlib.sha256()
for variable in [*reference.variables,*reference.optimizer.variables]:
    array=variable.numpy();fingerprint.update(str(array.dtype).encode());fingerprint.update(str(array.shape).encode());fingerprint.update(array.tobytes())
write_json(run/'modelo-guardado.json',dict(ruta=model_path.relative_to(ROOT).as_posix(),sha256=sha(model_path),
    estado_canonico_sha256=fingerprint.hexdigest(),optimizer_iterations=1440))
write_json(run/'lanzamientos.json',dict(vector_block=256,bn_parcial_block=[32,8,1],
    bn_parcial_grid=[1,384,1],bn_final_block=256,bn_final_grid=[48,1,1],
    softmax_block=32,softmax_grid=[128,1,1],ultimo_batch=20,recursos=compiled_resources(),
    fc_finish_regs=str(finish_fc.get_regs_per_thread()),
    gemm='cuBLAS SGEMM FP32 PEDANTIC; jerarquía interna de biblioteca, no fijada por el equipo'))
state('completada',pasos=1440)
stop.set();monitor.join(timeout=15)
if repetition==1:
    try:capture.wait(timeout=8)
    except subprocess.TimeoutExpired:capture.terminate();capture.wait();print('Captura no disponible; revisar capturador.log')
    capture_log.close()
display(pd.DataFrame(timings).fillna(''))
display(pd.DataFrame(epoch_rows))
print('MÉTRICAS REALES',json.dumps(metrics,indent=2,ensure_ascii=False))
fig,axes=plt.subplots(1,2,figsize=(10,3.6),layout='constrained')
for ax,keys,title in zip(axes,[('train_loss','val_loss'),('train_accuracy','val_accuracy')],['Pérdida','Exactitud']):
    for key in keys:ax.plot(range(1,16),[r[key] for r in epoch_rows],label=key)
    ax.set(xlabel='Época',ylabel=title,title=name+' · corrida '+str(repetition));ax.legend();ax.grid(alpha=.2)
fig.savefig(run/'curvas.png',dpi=160);fig.savefig(run/'curvas.svg');mostrar_figura(fig)
resource=pd.read_csv(run/'recursos.csv')
fig,axes=plt.subplots(2,1,figsize=(10,5),sharex=True,layout='constrained')
axes[0].plot(resource.transcurrido_s,resource.cpu_procesos_pct,color='#0072B2',label='CPU proceso e hijos (100% por núcleo)')
axes[1].plot(resource.transcurrido_s,resource.gpu_util_pct,color='#D55E00',label='Actividad GPU (%)')
axes[0].set(ylabel='Uso CPU (%)');axes[1].set(ylabel='Actividad GPU (%)',xlabel='Tiempo desde inicio del RUN (s)',ylim=(0,105))
for ax in axes:ax.legend(loc='upper right');ax.grid(alpha=.2)
fig.suptitle(name+' · monitorización real, corrida '+str(repetition))
fig.savefig(run/'recursos.png',dpi=160);fig.savefig(run/'recursos.svg');mostrar_figura(fig)
for picture in sorted(run.glob('captura-*.png')):display(Image(filename=str(picture)))
all_results.append(dict(run_id=run.name,**metrics));write_json(ROOT/'resultados/paralelo/EXP-012/progreso.json',all_results)
net.stream.synchronize();net.blas.close()
del net,reference,inputs,x,y,vx,vy,tx,ty,wx,wy,wv,wvy,wt,wty,p,op
tf.keras.backend.clear_session();gc.collect();cuda.current_context().deallocations.clear()


### FC-B: corrida 5

RUN: `RUN-20261118T134514Z-fc-b-05`. Inicialización/restauración → preparación CPU → entrenamiento CUDA → evaluación CPU y evidencias.

In [ ]:
name='FC-B';repetition=5;run_id='RUN-20261118T134514Z-fc-b-05'
run=ROOT/'resultados/paralelo/EXP-012'/run_id
run.mkdir(exist_ok=False)
timings=[];epoch_rows=[];events={};shuffle_hashes=[]
state('inicializacion')
stop,monitor=iniciar_monitor(run)
if repetition==1:
    capture_log=(run/'capturador.log').open('x',encoding='utf8')
    capture=subprocess.Popen([sys.executable,'-X','utf8',str(ROOT/'scripts/capturar_fc_real.py'),
        '--run',str(run),'--tipo','cuda','--modelo',name,'--nombre',name.lower()],
        stdout=capture_log,stderr=subprocess.STDOUT)
init_start=time.perf_counter()
tf,reference,seed,initial=fc_reference(name)
assert sha(initial)==config['estados_iniciales'][name]['sha256']
net=FCNetwork(reference,seed)
reference.summary()
print('GPU REAL',net.info,'Arena bytes',net.bytes_allocated,'BN momentum',[op['momentum'] for op in net.ops if op['kind']=='bn'])
write_json(run/'hardware.json',dict(device=net.info,tensorflow=tf.__version__,
    tf_build=tf.sysconfig.get_build_info(),cpu_tensor_device=tf.constant([1.]).device,
    tf_visible=[str(d) for d in tf.config.get_visible_devices()],cublas_version=net.blas.version,
    cublas_dll=str(net.blas.path),cublas_sha256=sha(net.blas.path),buffers_bytes=net.bytes_allocated,
    parametros=sum(p['value'].size for p in net.params),initial_checkpoint_sha256=sha(initial)))
assert not tf.config.get_visible_devices('GPU')
wx,wy=load_images(manifest['train'][:148],data_root,1)
wv,wvy=load_images(manifest['val'][:128],data_root,1)
wt,wty=load_images(manifest['test'][:32],data_root,1)
net.train_step(wx[:128],wy[:128],0,0);net.train_step(wx[128:],wy[128:],0,1)
reference.evaluate(wv,wvy,batch_size=128,verbose=0)
reference.evaluate(wt,wty,batch_size=32,verbose=0)
net.restore()
for p in net.params:
    assert np.array_equal(p['value'].copy_to_host(),p['initial'].ravel())
    assert not np.any(p['m'].copy_to_host()) and not np.any(p['v'].copy_to_host())
for op in net.ops:
    if op['kind']=='bn':
        assert np.array_equal(op['moving_mean'].copy_to_host(),op['initial_mean'])
        assert np.array_equal(op['moving_var'].copy_to_host(),op['initial_var'])
assert net.counter.copy_to_host()[0]==0
write_json(run/'warmup.json',dict(duracion_s=time.perf_counter()-init_start,train=148,val=128,test=32,
    restauracion_verificada=True,excluido_timing=True))
print('Warmup nuevo FC terminado y estado inicial restaurado; no incluido en entrenamiento.')


In [ ]:
state('preparacion_cpu',workers=6)
start=time.perf_counter()
inputs={split:load_images(manifest[split],data_root,6) for split in ['train','val','test']}
add('preparacion_cpu',time.perf_counter()-start)
tensors={}
for split,(images,labels) in inputs.items():
    a=hashlib.sha256(memoryview(images).cast('B')).hexdigest()
    b=hashlib.sha256(memoryview(labels).cast('B')).hexdigest()
    assert a==expected[split]['images_sha256'] and b==expected[split]['labels_sha256']
    tensors[split]=dict(images_sha256=a,labels_sha256=b,shape=list(images.shape),dtype=str(images.dtype))
write_json(run/'tensores.json',tensors)
x,y=inputs['train'];vx,vy=inputs['val'];tx,ty=inputs['test']
print('Preparación SOLO CPU (6 workers); bytes idénticos a EXP-002:',tensors)


In [ ]:
state('entrenamiento_'+name,epoca=1,pasos=0)
train_start_utc=utc();train_start=time.perf_counter_ns()
validations=0.;export_wall=0.
with net.timed('h2d_inicial'):net.restore()
total_events(net.drain_events())
rng=np.random.Generator(np.random.PCG64(seed))
for epoch in range(15):
    epoch_start=time.perf_counter()
    state('entrenamiento_'+name,epoca=epoch+1,pasos=net.steps)
    net.reset_metrics()
    order=rng.permutation(len(x))
    shuffle_hashes.append(hashlib.sha256(order.astype(np.int64).tobytes()).hexdigest())
    for j,begin in enumerate(range(0,len(x),128)):
        indices=order[begin:begin+128]
        total_events(net.train_step(x[indices],y[indices],epoch,j))
    export_start=time.perf_counter()
    extra,total=net.export();total_events(extra)
    export_wall+=time.perf_counter()-export_start
    assert int(total[2])==12180
    state('validacion_cpu',epoca=epoch+1,pasos=net.steps)
    val_start=time.perf_counter()
    val=reference.evaluate(vx,vy,batch_size=128,verbose=0,return_dict=True)
    val_s=time.perf_counter()-val_start;validations+=val_s
    epoch_rows.append(dict(modelo=name,repeticion=repetition,epoca=epoch+1,pasos_acumulados=net.steps,
        train_loss=float(total[0]/total[2]),train_accuracy=float(total[1]/total[2]),
        val_loss=float(val['loss']),val_accuracy=float(val['accuracy']),
        pared_epoca_s=time.perf_counter()-epoch_start,validacion_cpu_s=val_s))
    write_csv(run/'epocas.csv',epoch_rows)
    print('EPOCH',name,repetition,epoch_rows[-1],flush=True)
extra,_=net.export(optimizer=True);total_events(extra)
train_seconds=(time.perf_counter_ns()-train_start)/1e9
train_end_utc=utc()
assert net.steps==1440 and int(reference.optimizer.iterations.numpy())==1440
add('entrenamiento_'+name,train_seconds,inicio_utc=train_start_utc,fin_utc=train_end_utc)
add('validacion_anidada_'+name,validations)
for phase,seconds in events.items():add(phase,seconds,'evento_device')
print('ENTRENAMIENTO CUDA REAL',name,'corrida',repetition,'segundos',train_seconds,
    '| incluye staging, máscaras, H2D, GPU, D2H y validación CPU; pasos',net.steps)


In [ ]:
state('evaluacion_final_cpu',pasos=net.steps)
start=time.perf_counter_ns()
test=reference.evaluate(tx,ty,batch_size=32,verbose=0,return_dict=True)
eval_seconds=(time.perf_counter_ns()-start)/1e9
add('evaluacion_'+name,eval_seconds)
metrics=dict(modelo=name,repeticion=repetition,test={k:float(v) for k,v in test.items()},epocas=15,pasos=1440,
    entrenamiento_s=train_seconds,evaluacion_cpu_s=eval_seconds,
    comparabilidad='Datos, estado inicial, arquitectura y pasos equivalentes; CPU histórica; RNG interno diferente')
write_json(run/'metricas.json',metrics);write_csv(run/'tiempos.csv',timings)
write_json(run/'eventos.json',dict(eventos_s=events,validacion_cpu_s=validations,export_cpu_pared_s=export_wall,
    nota='Intervalos stream anidados; no equivalen a tiempo activo exclusivo de kernels; no sumar al tiempo pared'))
write_json(run/'ordenes.json',dict(seed=seed,generador='PCG64',epocas_sha256=shuffle_hashes))
param_count=sum(p['value'].size for p in net.params)
bn_count=sum(op['C']*2 for op in net.ops if op['kind']=='bn')
write_json(run/'transferencias.json',dict(lotes=net.transfers[-1440:],modelo_d2h_copias=16,
    modelo_bytes_por_copia=(param_count+bn_count)*4,optimizador_final_bytes=param_count*8+8,
    control_d2h_bytes=1440*4,reinicios_metricas_h2d_bytes=15*12,
    nota='Copia final adicional de pesos incluida; sin restar costes'))
state('guardado_fuera_timing')
model_dir=ROOT/'data/EXP-012'/run.name;model_dir.mkdir(parents=True,exist_ok=False)
model_path=model_dir/(name+'-CUDA.keras');reference.save(model_path)
fingerprint=hashlib.sha256()
for variable in [*reference.variables,*reference.optimizer.variables]:
    array=variable.numpy();fingerprint.update(str(array.dtype).encode());fingerprint.update(str(array.shape).encode());fingerprint.update(array.tobytes())
write_json(run/'modelo-guardado.json',dict(ruta=model_path.relative_to(ROOT).as_posix(),sha256=sha(model_path),
    estado_canonico_sha256=fingerprint.hexdigest(),optimizer_iterations=1440))
write_json(run/'lanzamientos.json',dict(vector_block=256,bn_parcial_block=[32,8,1],
    bn_parcial_grid=[1,384,1],bn_final_block=256,bn_final_grid=[48,1,1],
    softmax_block=32,softmax_grid=[128,1,1],ultimo_batch=20,recursos=compiled_resources(),
    fc_finish_regs=str(finish_fc.get_regs_per_thread()),
    gemm='cuBLAS SGEMM FP32 PEDANTIC; jerarquía interna de biblioteca, no fijada por el equipo'))
state('completada',pasos=1440)
stop.set();monitor.join(timeout=15)
if repetition==1:
    try:capture.wait(timeout=8)
    except subprocess.TimeoutExpired:capture.terminate();capture.wait();print('Captura no disponible; revisar capturador.log')
    capture_log.close()
display(pd.DataFrame(timings).fillna(''))
display(pd.DataFrame(epoch_rows))
print('MÉTRICAS REALES',json.dumps(metrics,indent=2,ensure_ascii=False))
fig,axes=plt.subplots(1,2,figsize=(10,3.6),layout='constrained')
for ax,keys,title in zip(axes,[('train_loss','val_loss'),('train_accuracy','val_accuracy')],['Pérdida','Exactitud']):
    for key in keys:ax.plot(range(1,16),[r[key] for r in epoch_rows],label=key)
    ax.set(xlabel='Época',ylabel=title,title=name+' · corrida '+str(repetition));ax.legend();ax.grid(alpha=.2)
fig.savefig(run/'curvas.png',dpi=160);fig.savefig(run/'curvas.svg');mostrar_figura(fig)
resource=pd.read_csv(run/'recursos.csv')
fig,axes=plt.subplots(2,1,figsize=(10,5),sharex=True,layout='constrained')
axes[0].plot(resource.transcurrido_s,resource.cpu_procesos_pct,color='#0072B2',label='CPU proceso e hijos (100% por núcleo)')
axes[1].plot(resource.transcurrido_s,resource.gpu_util_pct,color='#D55E00',label='Actividad GPU (%)')
axes[0].set(ylabel='Uso CPU (%)');axes[1].set(ylabel='Actividad GPU (%)',xlabel='Tiempo desde inicio del RUN (s)',ylim=(0,105))
for ax in axes:ax.legend(loc='upper right');ax.grid(alpha=.2)
fig.suptitle(name+' · monitorización real, corrida '+str(repetition))
fig.savefig(run/'recursos.png',dpi=160);fig.savefig(run/'recursos.svg');mostrar_figura(fig)
for picture in sorted(run.glob('captura-*.png')):display(Image(filename=str(picture)))
all_results.append(dict(run_id=run.name,**metrics));write_json(ROOT/'resultados/paralelo/EXP-012/progreso.json',all_results)
net.stream.synchronize();net.blas.close()
del net,reference,inputs,x,y,vx,vy,tx,ty,wx,wy,wv,wvy,wt,wty,p,op
tf.keras.backend.clear_session();gc.collect();cuda.current_context().deallocations.clear()


## Resumen de las diez corridas nuevas

No se mezclan las observaciones adicionales n=1 de otros notebooks con las campañas de cinco repeticiones.

In [ ]:
display(pd.DataFrame([{k:v for k,v in row.items() if k!='test'}|{'test_accuracy':row['test']['accuracy'],'test_loss':row['test']['loss']} for row in all_results]))
limits.__exit__(None,None,None)
print('FIN EXP-012: 150 épocas nuevas FC, cero entrenamientos CPU/CNN-B previos repetidos.')